In [4]:
import numpy as np
import pandas as pd
from scipy.optimize import minimize
from scipy.special import logsumexp


# ============================================================
# WCZYTANIE DANYCH
# ============================================================

def read_experiment_data_fig2d(csv_path):
    """
    Plik:
      - 4 wiersze = 4 wierzchołki
      - kolumny = obserwacje
    Zwraca:
      X shape = (n_obs, 4)
    """
    df = pd.read_csv(csv_path, header=None)
    df = df.dropna(axis=0, how="all").dropna(axis=1, how="all")

    X_raw = df.to_numpy()

    if X_raw.shape[0] != 4:
        raise ValueError(
            f"Oczekiwałem 4 wierszy w pliku {csv_path}, a jest {X_raw.shape[0]}."
        )

    uniq = np.unique(X_raw)
    if not np.all(np.isin(uniq, [0, 1])):
        raise ValueError(f"Dane muszą być binarne 0/1. Unikalne wartości: {uniq}")

    return X_raw.T.astype(int)


# ============================================================
# NARZĘDZIA
# ============================================================

def valid_rows_mask(X, edges):
    ok = np.ones(X.shape[0], dtype=bool)
    for u, v in edges:
        ok &= ~((X[:, u] == 1) & (X[:, v] == 1))
    return ok


def enumerate_independent_set_masks(p, edges):
    masks = []
    for mask in range(1 << p):
        good = True
        for u, v in edges:
            if ((mask >> u) & 1) and ((mask >> v) & 1):
                good = False
                break
        if good:
            masks.append(mask)
    return np.array(masks, dtype=np.int64)


def masks_to_matrix(masks, p):
    masks = np.asarray(masks, dtype=np.int64)
    return ((masks[:, None] >> np.arange(p, dtype=np.int64)) & 1).astype(int)


def matrix_to_masks(X):
    p = X.shape[1]
    powers = (1 << np.arange(p, dtype=np.int64))
    return (X.astype(np.int64) * powers).sum(axis=1)


def mask_to_bitstring(mask, p):
    return "".join(str((mask >> j) & 1) for j in range(p))


# ============================================================
# DOPASOWANIE multG(1,y)
# ============================================================

def fit_multG1_from_state_counts(
    state_counts,
    support_states,
    theta_init=None,
    theta_bounds=(-20.0, 20.0),
):
    counts = np.asarray(state_counts, dtype=float)
    S = np.asarray(support_states, dtype=float)

    n = counts.sum()
    if n <= 0:
        raise ValueError("Brak obserwacji w nośniku.")

    sufficient_stat = counts @ S
    p = S.shape[1]

    if theta_init is None:
        theta_init = np.zeros(p, dtype=float)
    else:
        theta_init = np.asarray(theta_init, dtype=float)

    def objective(theta):
        eta = S @ theta
        logZ = logsumexp(eta)
        probs = np.exp(eta - logZ)

        nll = n * logZ - sufficient_stat @ theta
        grad = n * (probs @ S) - sufficient_stat
        return nll, grad

    lo, hi = theta_bounds
    bounds = [(lo, hi)] * p

    opt = minimize(
        fun=lambda th: objective(th)[0],
        x0=theta_init,
        jac=lambda th: objective(th)[1],
        method="L-BFGS-B",
        bounds=bounds,
    )

    theta_hat = opt.x
    eta_hat = S @ theta_hat
    logZ_hat = logsumexp(eta_hat)
    probs_hat = np.exp(eta_hat - logZ_hat)

    return {
        "theta_hat": theta_hat,
        "y_hat": np.exp(theta_hat),
        "logZ_hat": float(logZ_hat),
        "loglik_hat": float(sufficient_stat @ theta_hat - n * logZ_hat),
        "probs_hat": probs_hat,
        "expected_state_counts": n * probs_hat,
        "fitted_marginals": probs_hat @ S,
        "optimizer_success": bool(opt.success),
        "optimizer_message": str(opt.message),
    }


def deviance_statistic(counts, probs):
    counts = np.asarray(counts, dtype=float)
    probs = np.asarray(probs, dtype=float)

    n = counts.sum()
    expected = n * probs

    mask = counts > 0
    return float(2.0 * np.sum(counts[mask] * np.log(counts[mask] / expected[mask])))


# ============================================================
# TEST: NAJPIERW FILTRACJA DO NOŚNIKA, POTEM TEST multG
# ============================================================

def test_filtered_multG1_P4_from_csv(
    csv_path,
    bootstrap_B=300,
    refit_in_bootstrap=True,
    seed=123,
):
    X = read_experiment_data_fig2d(csv_path)
    n_total = X.shape[0]

    # 1. wybór obserwacji z nośnika
    is_valid = valid_rows_mask(X, EDGES)
    X_valid = X[is_valid]
    n_valid = X_valid.shape[0]
    n_invalid = n_total - n_valid

    if n_valid == 0:
        raise ValueError("Po odrzuceniu obserwacji spoza nośnika nie zostały żadne dane.")

    # 2. exact support multG(1,y)
    support_masks = enumerate_independent_set_masks(P, EDGES)
    support_states = masks_to_matrix(support_masks, P)
    mask_to_index = {int(m): i for i, m in enumerate(support_masks.tolist())}

    # 3. liczniki stanów dla danych z nośnika
    valid_masks = matrix_to_masks(X_valid)
    obs_counts = np.zeros(len(support_masks), dtype=int)
    for m in valid_masks:
        obs_counts[mask_to_index[int(m)]] += 1

    # 4. dopasowanie multG(1,y)
    fit = fit_multG1_from_state_counts(
        state_counts=obs_counts,
        support_states=support_states,
    )

    D_obs = deviance_statistic(obs_counts, fit["probs_hat"])

    # 5. bootstrap parametryczny
    rng = np.random.default_rng(seed)
    D_boot = np.empty(bootstrap_B, dtype=float)

    for b in range(bootstrap_B):
        boot_counts = rng.multinomial(n_valid, fit["probs_hat"])

        if refit_in_bootstrap:
            fit_b = fit_multG1_from_state_counts(
                state_counts=boot_counts,
                support_states=support_states,
                theta_init=fit["theta_hat"],
            )
            D_boot[b] = deviance_statistic(boot_counts, fit_b["probs_hat"])
        else:
            D_boot[b] = deviance_statistic(boot_counts, fit["probs_hat"])

    p_value = (1.0 + np.sum(D_boot >= D_obs)) / (bootstrap_B + 1.0)

    state_table = pd.DataFrame({
        "state_mask": support_masks,
        "state": [mask_to_bitstring(int(m), P) for m in support_masks],
        "obs": obs_counts,
        "exp": fit["expected_state_counts"],
        "prob_hat": fit["probs_hat"],
    }).sort_values(["obs", "exp"], ascending=[False, False]).reset_index(drop=True)

    marginal_table = pd.DataFrame({
        "vertex": [f"v{j}" for j in range(P)],
        "obs_mean_filtered": X_valid.mean(axis=0),
        "fit_mean": fit["fitted_marginals"],
        "diff": X_valid.mean(axis=0) - fit["fitted_marginals"],
        "y_hat": fit["y_hat"],
        "theta_hat": fit["theta_hat"],
    })

    return {
        "summary": {
            "csv_path": csv_path,
            "graph": "P4 (path of length 3)",
            "edges": EDGES,
            "n_total": int(n_total),
            "n_valid": int(n_valid),
            "n_invalid": int(n_invalid),
            "frac_valid": float(n_valid / n_total),
            "support_size": int(len(support_masks)),
            "deviance_obs": float(D_obs),
            "bootstrap_p_value": float(p_value),
            "reject_multG_at_05": bool(p_value < 0.05),
            "optimizer_success": bool(fit["optimizer_success"]),
            "optimizer_message": fit["optimizer_message"],
        },
        "fit": fit,
        "state_table": state_table,
        "marginal_table": marginal_table,
        "X_valid": X_valid,
        "valid_rows_mask": is_valid,
        "support_states": support_states,
        "support_masks": support_masks,
        "obs_counts": obs_counts,
        "D_boot": D_boot,
    }


def print_report_filtered_P4(result, top_k=20, digits=4):
    s = result["summary"]

    print("=== TEST multG(1,y) PO OGRANICZENIU DO NOŚNIKA ===")
    print(f"csv_path             = {s['csv_path']}")
    print(f"graph                = {s['graph']}")
    print(f"edges                = {s['edges']}")
    print(f"n_total              = {s['n_total']}")
    print(f"n_valid              = {s['n_valid']}")
    print(f"n_invalid            = {s['n_invalid']}")
    print(f"frac_valid           = {s['frac_valid']:.{digits}f}")
    print(f"support_size         = {s['support_size']}")
    print(f"deviance_obs         = {s['deviance_obs']:.{digits}f}")
    print(f"bootstrap_p_value    = {s['bootstrap_p_value']:.{digits}f}")
    print(f"reject_multG_at_05   = {s['reject_multG_at_05']}")
    print(f"optimizer_success    = {s['optimizer_success']}")
    print(f"optimizer_message    = {s['optimizer_message']}")
    print()

    print("=== MARGINALS DLA DANYCH Z NOŚNIKA ===")
    print(result["marginal_table"].round(digits).to_string(index=False))
    print()

    print(f"=== TOP {top_k} STANÓW Z NOŚNIKA ===")
    print(result["state_table"].head(top_k).round(digits).to_string(index=False))




In [5]:
# ============================================================
# USTAWIENIA
# ============================================================

CSV_PATH = "experiment_data_fig2d.csv"

# ścieżka długości 3: 0-1-2-3
EDGES = [
    (0, 1),
    (1, 2),
    (2, 3),
]

P = 4
BOOTSTRAP_B = 500
REFIT_IN_BOOTSTRAP = True
SEED = 123

# ============================================================
# URUCHOMIENIE
# ============================================================

result = test_filtered_multG1_P4_from_csv(
    csv_path=CSV_PATH,
    bootstrap_B=BOOTSTRAP_B,
    refit_in_bootstrap=REFIT_IN_BOOTSTRAP,
    seed=SEED,
)

print_report_filtered_P4(result, top_k=20, digits=4)


=== TEST multG(1,y) PO OGRANICZENIU DO NOŚNIKA ===
csv_path             = experiment_data_fig2d.csv
graph                = P4 (path of length 3)
edges                = [(0, 1), (1, 2), (2, 3)]
n_total              = 678
n_valid              = 191
n_invalid            = 487
frac_valid           = 0.2817
support_size         = 8
deviance_obs         = 0.5081
bootstrap_p_value    = 0.9102
reject_multG_at_05   = False
optimizer_success    = True
optimizer_message    = CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*EPSMCH

=== MARGINALS DLA DANYCH Z NOŚNIKA ===
vertex  obs_mean_filtered  fit_mean  diff   y_hat  theta_hat
    v0             0.4398    0.4398  -0.0  3.1111     1.1350
    v1             0.4188    0.4188   0.0 11.3410     2.4284
    v2             0.4293    0.4293  -0.0 14.0093     2.6397
    v3             0.4555    0.4555   0.0  3.9545     1.3749

=== TOP 20 STANÓW Z NOŚNIKA ===
 state_mask state  obs     exp  prob_hat
         10  0101   63 63.8531    0.3343
          5  1010 

=== TEST multG(1,y) PO OGRANICZENIU DO NOŚNIKA ===
csv_path             = experiment_data_fig2j.csv
graph                = P4 (path of length 3)
edges                = [(0, 2), (1, 2), (2, 3)]
n_total              = 425
n_valid              = 183
n_invalid            = 242
frac_valid           = 0.4306
support_size         = 9
deviance_obs         = 8.9469
bootstrap_p_value    = 0.0918
reject_multG_at_05   = False
optimizer_success    = True
optimizer_message    = CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*EPSMCH

=== MARGINALS DLA DANYCH Z NOŚNIKA ===
vertex  obs_mean_filtered  fit_mean  diff   y_hat  theta_hat
    v0             0.2022    0.2022  -0.0  1.7619     0.5664
    v1             0.2459    0.2459  -0.0  3.4615     1.2417
    v2             0.6831    0.6831   0.0 77.0145     4.3440
    v3             0.2077    0.2077   0.0  1.9000     0.6419

=== TOP 20 STANÓW Z NOŚNIKA ===
 state_mask state  obs      exp  prob_hat
          4  0010  125 124.9999    0.6831
         11  110

In [7]:
# ============================================================
# USTAWIENIA
# ============================================================

CSV_PATH = "experiment_data_fig2j.csv"


EDGES = [
    (0, 1),  # 1--2
    (0, 2),  # 1--3
    (1, 2),  # 2--3
    (2, 3),  # 3--4
]

P = 4
BOOTSTRAP_B = 500
REFIT_IN_BOOTSTRAP = True
SEED = 123

# ============================================================
# URUCHOMIENIE
# ============================================================

result = test_filtered_multG1_P4_from_csv(
    csv_path=CSV_PATH,
    bootstrap_B=BOOTSTRAP_B,
    refit_in_bootstrap=REFIT_IN_BOOTSTRAP,
    seed=SEED,
)

print_report_filtered_P4(result, top_k=20, digits=4)


=== TEST multG(1,y) PO OGRANICZENIU DO NOŚNIKA ===
csv_path             = experiment_data_fig2j.csv
graph                = P4 (path of length 3)
edges                = [(0, 1), (0, 2), (1, 2), (2, 3)]
n_total              = 425
n_valid              = 155
n_invalid            = 270
frac_valid           = 0.3647
support_size         = 7
deviance_obs         = 6.5478
bootstrap_p_value    = 0.0798
reject_multG_at_05   = False
optimizer_success    = True
optimizer_message    = CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*EPSMCH

=== MARGINALS DLA DANYCH Z NOŚNIKA ===
vertex  obs_mean_filtered  fit_mean  diff   y_hat  theta_hat
    v0             0.0581    0.0581   0.0  2.2501     0.8110
    v1             0.1097    0.1097   0.0  4.2502     1.4470
    v2             0.8065    0.8065  -0.0 72.1206     4.2783
    v3             0.1097    0.1097   0.0  1.3077     0.2683

=== TOP 20 STANÓW Z NOŚNIKA ===
 state_mask state  obs      exp  prob_hat
          4  0010  125 125.0005    0.8065
        

In [11]:
import numpy as np
import pandas as pd
from scipy.optimize import minimize
from scipy.special import logsumexp


# ============================================================
# USTAWIENIA
# ============================================================

CSV_PATH = "experiment_data_fig3e.csv"   # zmień nazwę pliku, jeśli teraz masz inny

# Graf z obrazka: K5
# Zakładamy, że wiersze pliku odpowiadają kolejno wierzchołkom 1,2,3,4,5
EDGES = [
    (0, 1), (0, 2), (0, 3), (0, 4),
    (1, 2), (1, 3), (1, 4),
    (2, 3), (2, 4),
    (3, 4),
]

P = 5
BOOTSTRAP_B = 500
REFIT_IN_BOOTSTRAP = True
SEED = 123


# ============================================================
# WCZYTANIE DANYCH
# ============================================================

def read_experiment_data(csv_path):
    """
    Plik:
      - 5 wierszy = 5 wierzchołków
      - kolumny = obserwacje
    Zwraca:
      X shape = (n_obs, 5)
    """
    df = pd.read_csv(csv_path, header=None)
    df = df.dropna(axis=0, how="all").dropna(axis=1, how="all")
    df = df.iloc[:5, :]
    X_raw = df.to_numpy()

    if X_raw.shape[0] != 5:
        raise ValueError(
            f"Oczekiwałem 5 wierszy w pliku {csv_path}, a jest {X_raw.shape[0]}."
        )

    uniq = np.unique(X_raw)
    if not np.all(np.isin(uniq, [0, 1])):
        raise ValueError(f"Dane muszą być binarne 0/1. Unikalne wartości: {uniq}")

    return X_raw.T.astype(int)


# ============================================================
# NARZĘDZIA
# ============================================================

def valid_rows_mask(X, edges):
    ok = np.ones(X.shape[0], dtype=bool)
    for u, v in edges:
        ok &= ~((X[:, u] == 1) & (X[:, v] == 1))
    return ok


def enumerate_independent_set_masks(p, edges):
    masks = []
    for mask in range(1 << p):
        good = True
        for u, v in edges:
            if ((mask >> u) & 1) and ((mask >> v) & 1):
                good = False
                break
        if good:
            masks.append(mask)
    return np.array(masks, dtype=np.int64)


def masks_to_matrix(masks, p):
    masks = np.asarray(masks, dtype=np.int64)
    return ((masks[:, None] >> np.arange(p, dtype=np.int64)) & 1).astype(int)


def matrix_to_masks(X):
    p = X.shape[1]
    powers = (1 << np.arange(p, dtype=np.int64))
    return (X.astype(np.int64) * powers).sum(axis=1)


def mask_to_bitstring(mask, p):
    return "".join(str((mask >> j) & 1) for j in range(p))


# ============================================================
# DOPASOWANIE multG(1,y)
# ============================================================

def fit_multG1_from_state_counts(
    state_counts,
    support_states,
    theta_init=None,
    theta_bounds=(-20.0, 20.0),
):
    counts = np.asarray(state_counts, dtype=float)
    S = np.asarray(support_states, dtype=float)

    n = counts.sum()
    if n <= 0:
        raise ValueError("Brak obserwacji w nośniku.")

    sufficient_stat = counts @ S
    p = S.shape[1]

    if theta_init is None:
        theta_init = np.zeros(p, dtype=float)
    else:
        theta_init = np.asarray(theta_init, dtype=float)

    def objective(theta):
        eta = S @ theta
        logZ = logsumexp(eta)
        probs = np.exp(eta - logZ)

        nll = n * logZ - sufficient_stat @ theta
        grad = n * (probs @ S) - sufficient_stat
        return nll, grad

    lo, hi = theta_bounds
    bounds = [(lo, hi)] * p

    opt = minimize(
        fun=lambda th: objective(th)[0],
        x0=theta_init,
        jac=lambda th: objective(th)[1],
        method="L-BFGS-B",
        bounds=bounds,
    )

    theta_hat = opt.x
    eta_hat = S @ theta_hat
    logZ_hat = logsumexp(eta_hat)
    probs_hat = np.exp(eta_hat - logZ_hat)

    return {
        "theta_hat": theta_hat,
        "y_hat": np.exp(theta_hat),
        "logZ_hat": float(logZ_hat),
        "loglik_hat": float(sufficient_stat @ theta_hat - n * logZ_hat),
        "probs_hat": probs_hat,
        "expected_state_counts": n * probs_hat,
        "fitted_marginals": probs_hat @ S,
        "optimizer_success": bool(opt.success),
        "optimizer_message": str(opt.message),
    }


def deviance_statistic(counts, probs):
    counts = np.asarray(counts, dtype=float)
    probs = np.asarray(probs, dtype=float)

    n = counts.sum()
    expected = n * probs

    mask = counts > 0
    return float(2.0 * np.sum(counts[mask] * np.log(counts[mask] / expected[mask])))


# ============================================================
# TEST: WYBIERAMY OBSERWACJE Z NOŚNIKA I TESTUJEMY multG(1,y)
# ============================================================

def test_filtered_multG1_K5_from_csv(
    csv_path,
    bootstrap_B=300,
    refit_in_bootstrap=True,
    seed=123,
):
    X = read_experiment_data(csv_path)
    n_total = X.shape[0]

    # 1. wybór obserwacji z nośnika
    is_valid = valid_rows_mask(X, EDGES)
    X_valid = X[is_valid]
    n_valid = X_valid.shape[0]
    n_invalid = n_total - n_valid

    if n_valid == 0:
        raise ValueError("Po odrzuceniu obserwacji spoza nośnika nie zostały żadne dane.")

    # 2. exact support multG(1,y)
    support_masks = enumerate_independent_set_masks(P, EDGES)
    support_states = masks_to_matrix(support_masks, P)
    mask_to_index = {int(m): i for i, m in enumerate(support_masks.tolist())}

    # 3. liczniki stanów dla danych z nośnika
    valid_masks = matrix_to_masks(X_valid)
    obs_counts = np.zeros(len(support_masks), dtype=int)
    for m in valid_masks:
        obs_counts[mask_to_index[int(m)]] += 1

    # 4. dopasowanie multG(1,y)
    fit = fit_multG1_from_state_counts(
        state_counts=obs_counts,
        support_states=support_states,
    )

    D_obs = deviance_statistic(obs_counts, fit["probs_hat"])

    # 5. bootstrap parametryczny
    rng = np.random.default_rng(seed)
    D_boot = np.empty(bootstrap_B, dtype=float)

    for b in range(bootstrap_B):
        boot_counts = rng.multinomial(n_valid, fit["probs_hat"])

        if refit_in_bootstrap:
            fit_b = fit_multG1_from_state_counts(
                state_counts=boot_counts,
                support_states=support_states,
                theta_init=fit["theta_hat"],
            )
            D_boot[b] = deviance_statistic(boot_counts, fit_b["probs_hat"])
        else:
            D_boot[b] = deviance_statistic(boot_counts, fit["probs_hat"])

    p_value = (1.0 + np.sum(D_boot >= D_obs)) / (bootstrap_B + 1.0)

    state_table = pd.DataFrame({
        "state_mask": support_masks,
        "state": [mask_to_bitstring(int(m), P) for m in support_masks],
        "obs": obs_counts,
        "exp": fit["expected_state_counts"],
        "prob_hat": fit["probs_hat"],
    }).sort_values(["obs", "exp"], ascending=[False, False]).reset_index(drop=True)

    marginal_table = pd.DataFrame({
        "vertex": ["1", "2", "3", "4", "5"],
        "obs_mean_filtered": X_valid.mean(axis=0),
        "fit_mean": fit["fitted_marginals"],
        "diff": X_valid.mean(axis=0) - fit["fitted_marginals"],
        "y_hat": fit["y_hat"],
        "theta_hat": fit["theta_hat"],
    })

    return {
        "summary": {
            "csv_path": csv_path,
            "graph": "K5",
            "edges_python": EDGES,
            "n_total": int(n_total),
            "n_valid": int(n_valid),
            "n_invalid": int(n_invalid),
            "frac_valid": float(n_valid / n_total),
            "support_size": int(len(support_masks)),
            "deviance_obs": float(D_obs),
            "bootstrap_p_value": float(p_value),
            "reject_multG_at_05": bool(p_value < 0.05),
            "optimizer_success": bool(fit["optimizer_success"]),
            "optimizer_message": fit["optimizer_message"],
        },
        "fit": fit,
        "state_table": state_table,
        "marginal_table": marginal_table,
        "X_valid": X_valid,
        "valid_rows_mask": is_valid,
        "support_states": support_states,
        "support_masks": support_masks,
        "obs_counts": obs_counts,
        "D_boot": D_boot,
    }


def print_report_K5(result, top_k=20, digits=4):
    s = result["summary"]

    print("=== TEST multG(1,y) PO OGRANICZENIU DO NOŚNIKA ===")
    print(f"csv_path             = {s['csv_path']}")
    print(f"graph                = {s['graph']}")
    print(f"edges_python         = {s['edges_python']}")
    print(f"n_total              = {s['n_total']}")
    print(f"n_valid              = {s['n_valid']}")
    print(f"n_invalid            = {s['n_invalid']}")
    print(f"frac_valid           = {s['frac_valid']:.4f}")
    print(f"support_size         = {s['support_size']}")
    print(f"deviance_obs         = {s['deviance_obs']:.4f}")
    print(f"bootstrap_p_value    = {s['bootstrap_p_value']:.4f}")
    print(f"reject_multG_at_05   = {s['reject_multG_at_05']}")
    print(f"optimizer_success    = {s['optimizer_success']}")
    print(f"optimizer_message    = {s['optimizer_message']}")
    print()

    print("=== MARGINALS DLA DANYCH Z NOŚNIKA ===")
    print(result["marginal_table"].round(digits).to_string(index=False))
    print()

    print(f"=== TOP {top_k} STANÓW Z NOŚNIKA ===")
    print(result["state_table"].head(top_k).round(digits).to_string(index=False))


# ============================================================
# URUCHOMIENIE
# ============================================================

result = test_filtered_multG1_K5_from_csv(
    csv_path=CSV_PATH,
    bootstrap_B=BOOTSTRAP_B,
    refit_in_bootstrap=REFIT_IN_BOOTSTRAP,
    seed=SEED,
)

print_report_K5(result, top_k=20, digits=4)

=== TEST multG(1,y) PO OGRANICZENIU DO NOŚNIKA ===
csv_path             = experiment_data_fig3e.csv
graph                = K5
edges_python         = [(0, 1), (0, 2), (0, 3), (0, 4), (1, 2), (1, 3), (1, 4), (2, 3), (2, 4), (3, 4)]
n_total              = 1961
n_valid              = 9
n_invalid            = 1952
frac_valid           = 0.0046
support_size         = 6
deviance_obs         = 0.0000
bootstrap_p_value    = 0.1756
reject_multG_at_05   = False
optimizer_success    = True
optimizer_message    = CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*EPSMCH

=== MARGINALS DLA DANYCH Z NOŚNIKA ===
vertex  obs_mean_filtered  fit_mean  diff        y_hat  theta_hat
     1             0.0000    0.0000  -0.0 0.000000e+00   -11.1553
     2             0.0000    0.0000  -0.0 0.000000e+00   -11.1553
     3             0.3333    0.3333   0.0 2.582285e+07    17.0668
     4             0.4444    0.4445  -0.0 3.443134e+07    17.3545
     5             0.2222    0.2222   0.0 1.721469e+07    16.6613

=== 

In [14]:
import numpy as np
import pandas as pd
from scipy.optimize import minimize
from scipy.special import logsumexp


# ============================================================
# USTAWIENIA
# ============================================================

CSV_PATH = "experiment_data_fig2l.csv"

# Graf na 4 wierzchołkach:
# 1--3, 2--3, 3--4
#
# W indeksacji pythonowej:
# 1 -> 0
# 2 -> 1
# 3 -> 2
# 4 -> 3
EDGES = [
    (0, 2),  # 1--3
    (2, 3),  # 3--4
    (1, 2),  # 2--3
]

P = 4
BOOTSTRAP_B = 500
REFIT_IN_BOOTSTRAP = True
SEED = 123


# ============================================================
# WCZYTANIE DANYCH
# ============================================================

def read_experiment_data(csv_path):
    """
    Plik:
      - co najmniej 4 wiersze = wierzchołki
      - kolumny = obserwacje

    Bierzemy pierwsze 4 wiersze i transponujemy.
    """
    df = pd.read_csv(csv_path, header=None)
    df = df.dropna(axis=0, how="all").dropna(axis=1, how="all")

    if df.shape[0] < 4:
        raise ValueError(
            f"Oczekiwałem co najmniej 4 wierszy w pliku {csv_path}, a jest {df.shape[0]}."
        )

    X_raw = df.iloc[:4, :].to_numpy()

    uniq = np.unique(X_raw)
    if not np.all(np.isin(uniq, [0, 1])):
        raise ValueError(f"Dane muszą być binarne 0/1. Unikalne wartości: {uniq}")

    return X_raw.T.astype(int)


# ============================================================
# NARZĘDZIA
# ============================================================

def valid_rows_mask(X, edges):
    ok = np.ones(X.shape[0], dtype=bool)
    for u, v in edges:
        ok &= ~((X[:, u] == 1) & (X[:, v] == 1))
    return ok


def enumerate_independent_set_masks(p, edges):
    masks = []
    for mask in range(1 << p):
        good = True
        for u, v in edges:
            if ((mask >> u) & 1) and ((mask >> v) & 1):
                good = False
                break
        if good:
            masks.append(mask)
    return np.array(masks, dtype=np.int64)


def masks_to_matrix(masks, p):
    masks = np.asarray(masks, dtype=np.int64)
    return ((masks[:, None] >> np.arange(p, dtype=np.int64)) & 1).astype(int)


def matrix_to_masks(X):
    p = X.shape[1]
    powers = (1 << np.arange(p, dtype=np.int64))
    return (X.astype(np.int64) * powers).sum(axis=1)


def mask_to_bitstring(mask, p):
    return "".join(str((mask >> j) & 1) for j in range(p))


# ============================================================
# DOPASOWANIE multG(1,y)
# ============================================================

def fit_multG1_from_state_counts(
    state_counts,
    support_states,
    theta_init=None,
    theta_bounds=(-20.0, 20.0),
):
    counts = np.asarray(state_counts, dtype=float)
    S = np.asarray(support_states, dtype=float)

    n = counts.sum()
    if n <= 0:
        raise ValueError("Brak obserwacji w nośniku.")

    sufficient_stat = counts @ S
    p = S.shape[1]

    if theta_init is None:
        theta_init = np.zeros(p, dtype=float)
    else:
        theta_init = np.asarray(theta_init, dtype=float)

    def objective(theta):
        eta = S @ theta
        logZ = logsumexp(eta)
        probs = np.exp(eta - logZ)

        nll = n * logZ - sufficient_stat @ theta
        grad = n * (probs @ S) - sufficient_stat
        return nll, grad

    lo, hi = theta_bounds
    bounds = [(lo, hi)] * p

    opt = minimize(
        fun=lambda th: objective(th)[0],
        x0=theta_init,
        jac=lambda th: objective(th)[1],
        method="L-BFGS-B",
        bounds=bounds,
    )

    theta_hat = opt.x
    eta_hat = S @ theta_hat
    logZ_hat = logsumexp(eta_hat)
    probs_hat = np.exp(eta_hat - logZ_hat)

    return {
        "theta_hat": theta_hat,
        "y_hat": np.exp(theta_hat),
        "logZ_hat": float(logZ_hat),
        "loglik_hat": float(sufficient_stat @ theta_hat - n * logZ_hat),
        "probs_hat": probs_hat,
        "expected_state_counts": n * probs_hat,
        "fitted_marginals": probs_hat @ S,
        "optimizer_success": bool(opt.success),
        "optimizer_message": str(opt.message),
    }


def deviance_statistic(counts, probs):
    counts = np.asarray(counts, dtype=float)
    probs = np.asarray(probs, dtype=float)

    n = counts.sum()
    expected = n * probs

    mask = counts > 0
    return float(2.0 * np.sum(counts[mask] * np.log(counts[mask] / expected[mask])))


# ============================================================
# TEST: WYBIERAMY OBSERWACJE Z NOŚNIKA I TESTUJEMY multG(1,y)
# ============================================================

def test_filtered_multG1_G_from_csv(
    csv_path,
    bootstrap_B=300,
    refit_in_bootstrap=True,
    seed=123,
):
    X = read_experiment_data(csv_path)
    n_total = X.shape[0]

    is_valid = valid_rows_mask(X, EDGES)
    X_valid = X[is_valid]
    n_valid = X_valid.shape[0]
    n_invalid = n_total - n_valid

    if n_valid == 0:
        raise ValueError("Po odrzuceniu obserwacji spoza nośnika nie zostały żadne dane.")

    support_masks = enumerate_independent_set_masks(P, EDGES)
    support_states = masks_to_matrix(support_masks, P)
    mask_to_index = {int(m): i for i, m in enumerate(support_masks.tolist())}

    valid_masks = matrix_to_masks(X_valid)
    obs_counts = np.zeros(len(support_masks), dtype=int)
    for m in valid_masks:
        obs_counts[mask_to_index[int(m)]] += 1

    fit = fit_multG1_from_state_counts(
        state_counts=obs_counts,
        support_states=support_states,
    )

    D_obs = deviance_statistic(obs_counts, fit["probs_hat"])

    rng = np.random.default_rng(seed)
    D_boot = np.empty(bootstrap_B, dtype=float)

    for b in range(bootstrap_B):
        boot_counts = rng.multinomial(n_valid, fit["probs_hat"])

        if refit_in_bootstrap:
            fit_b = fit_multG1_from_state_counts(
                state_counts=boot_counts,
                support_states=support_states,
                theta_init=fit["theta_hat"],
            )
            D_boot[b] = deviance_statistic(boot_counts, fit_b["probs_hat"])
        else:
            D_boot[b] = deviance_statistic(boot_counts, fit["probs_hat"])

    p_value = (1.0 + np.sum(D_boot >= D_obs)) / (bootstrap_B + 1.0)

    state_table = pd.DataFrame({
        "state_mask": support_masks,
        "state": [mask_to_bitstring(int(m), P) for m in support_masks],
        "obs": obs_counts,
        "exp": fit["expected_state_counts"],
        "prob_hat": fit["probs_hat"],
    }).sort_values(["obs", "exp"], ascending=[False, False]).reset_index(drop=True)

    marginal_table = pd.DataFrame({
        "vertex": ["1", "2", "3", "4"],
        "obs_mean_filtered": X_valid.mean(axis=0),
        "fit_mean": fit["fitted_marginals"],
        "diff": X_valid.mean(axis=0) - fit["fitted_marginals"],
        "y_hat": fit["y_hat"],
        "theta_hat": fit["theta_hat"],
    })

    return {
        "summary": {
            "csv_path": csv_path,
            "graph": "edges (1,3), (2,3), (3,4)",
            "edges_python": EDGES,
            "n_total": int(n_total),
            "n_valid": int(n_valid),
            "n_invalid": int(n_invalid),
            "frac_valid": float(n_valid / n_total),
            "support_size": int(len(support_masks)),
            "deviance_obs": float(D_obs),
            "bootstrap_p_value": float(p_value),
            "reject_multG_at_05": bool(p_value < 0.05),
            "optimizer_success": bool(fit["optimizer_success"]),
            "optimizer_message": fit["optimizer_message"],
        },
        "fit": fit,
        "state_table": state_table,
        "marginal_table": marginal_table,
        "X_valid": X_valid,
        "valid_rows_mask": is_valid,
        "support_states": support_states,
        "support_masks": support_masks,
        "obs_counts": obs_counts,
        "D_boot": D_boot,
    }


def print_report_G(result, top_k=20, digits=4):
    s = result["summary"]

    print("=== TEST multG(1,y) PO OGRANICZENIU DO NOŚNIKA ===")
    print(f"csv_path             = {s['csv_path']}")
    print(f"graph                = {s['graph']}")
    print(f"edges_python         = {s['edges_python']}")
    print(f"n_total              = {s['n_total']}")
    print(f"n_valid              = {s['n_valid']}")
    print(f"n_invalid            = {s['n_invalid']}")
    print(f"frac_valid           = {s['frac_valid']:.4f}")
    print(f"support_size         = {s['support_size']}")
    print(f"deviance_obs         = {s['deviance_obs']:.4f}")
    print(f"bootstrap_p_value    = {s['bootstrap_p_value']:.4f}")
    print(f"reject_multG_at_05   = {s['reject_multG_at_05']}")
    print(f"optimizer_success    = {s['optimizer_success']}")
    print(f"optimizer_message    = {s['optimizer_message']}")
    print()

    print("=== MARGINALS DLA DANYCH Z NOŚNIKA ===")
    print(result["marginal_table"].round(digits).to_string(index=False))
    print()

    print(f"=== TOP {top_k} STANÓW Z NOŚNIKA ===")
    print(result["state_table"].head(top_k).round(digits).to_string(index=False))


# ============================================================
# URUCHOMIENIE
# ============================================================

result = test_filtered_multG1_G_from_csv(
    csv_path=CSV_PATH,
    bootstrap_B=BOOTSTRAP_B,
    refit_in_bootstrap=REFIT_IN_BOOTSTRAP,
    seed=SEED,
)

print_report_G(result, top_k=20, digits=4)

=== TEST multG(1,y) PO OGRANICZENIU DO NOŚNIKA ===
csv_path             = experiment_data_fig2l.csv
graph                = edges (1,3), (2,3), (3,4)
edges_python         = [(0, 2), (2, 3), (1, 2)]
n_total              = 570
n_valid              = 185
n_invalid            = 385
frac_valid           = 0.3246
support_size         = 9
deviance_obs         = 6.3247
bootstrap_p_value    = 0.2295
reject_multG_at_05   = False
optimizer_success    = True
optimizer_message    = CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*EPSMCH

=== MARGINALS DLA DANYCH Z NOŚNIKA ===
vertex  obs_mean_filtered  fit_mean  diff   y_hat  theta_hat
     1             0.4216    0.4216  -0.0  3.5455     1.2657
     2             0.4162    0.4162   0.0  3.3478     1.2083
     3             0.4595    0.4595   0.0 41.9960     3.7376
     4             0.3243    0.3243   0.0  1.5000     0.4055

=== TOP 20 STANÓW Z NOŚNIKA ===
 state_mask state  obs    exp  prob_hat
          4  0010   85 85.000    0.4595
         11  110

In [15]:
import numpy as np
import pandas as pd
from scipy.optimize import minimize
from scipy.special import logsumexp


# ============================================================
# USTAWIENIA
# ============================================================

CSV_PATH = "experiment_data_fig4d.csv"   # zmień nazwę pliku, jeśli trzeba

# Graf z obrazka: gwiazda z centrum w 1
# Zakładamy, że wiersze pliku odpowiadają kolejno wierzchołkom:
# 1,2,3,4,5,6,7
EDGES = [
    (0, 1),  # 1--2
    (0, 2),  # 1--3
    (0, 3),  # 1--4
    (0, 4),  # 1--5
    (0, 5),  # 1--6
    (0, 6),  # 1--7
]

P = 7
BOOTSTRAP_B = 500
REFIT_IN_BOOTSTRAP = True
SEED = 123


# ============================================================
# WCZYTANIE DANYCH
# ============================================================

def read_experiment_data(csv_path):
    """
    Plik:
      - co najmniej 7 wierszy = wierzchołki
      - kolumny = obserwacje

    Bierzemy pierwsze 7 wierszy i transponujemy.
    """
    df = pd.read_csv(csv_path, header=None)
    df = df.dropna(axis=0, how="all").dropna(axis=1, how="all")
    df = df.iloc[:7,]
    if df.shape[0] < 7:
        raise ValueError(
            f"Oczekiwałem co najmniej 7 wierszy w pliku {csv_path}, a jest {df.shape[0]}."
        )

    X_raw = df.iloc[:7, :].to_numpy()

    uniq = np.unique(X_raw)
    if not np.all(np.isin(uniq, [0, 1])):
        raise ValueError(f"Dane muszą być binarne 0/1. Unikalne wartości: {uniq}")

    return X_raw.T.astype(int)


# ============================================================
# NARZĘDZIA
# ============================================================

def valid_rows_mask(X, edges):
    ok = np.ones(X.shape[0], dtype=bool)
    for u, v in edges:
        ok &= ~((X[:, u] == 1) & (X[:, v] == 1))
    return ok


def enumerate_independent_set_masks(p, edges):
    masks = []
    for mask in range(1 << p):
        good = True
        for u, v in edges:
            if ((mask >> u) & 1) and ((mask >> v) & 1):
                good = False
                break
        if good:
            masks.append(mask)
    return np.array(masks, dtype=np.int64)


def masks_to_matrix(masks, p):
    masks = np.asarray(masks, dtype=np.int64)
    return ((masks[:, None] >> np.arange(p, dtype=np.int64)) & 1).astype(int)


def matrix_to_masks(X):
    p = X.shape[1]
    powers = (1 << np.arange(p, dtype=np.int64))
    return (X.astype(np.int64) * powers).sum(axis=1)


def mask_to_bitstring(mask, p):
    return "".join(str((mask >> j) & 1) for j in range(p))


# ============================================================
# DOPASOWANIE multG(1,y)
# ============================================================

def fit_multG1_from_state_counts(
    state_counts,
    support_states,
    theta_init=None,
    theta_bounds=(-20.0, 20.0),
):
    counts = np.asarray(state_counts, dtype=float)
    S = np.asarray(support_states, dtype=float)

    n = counts.sum()
    if n <= 0:
        raise ValueError("Brak obserwacji w nośniku.")

    sufficient_stat = counts @ S
    p = S.shape[1]

    if theta_init is None:
        theta_init = np.zeros(p, dtype=float)
    else:
        theta_init = np.asarray(theta_init, dtype=float)

    def objective(theta):
        eta = S @ theta
        logZ = logsumexp(eta)
        probs = np.exp(eta - logZ)

        nll = n * logZ - sufficient_stat @ theta
        grad = n * (probs @ S) - sufficient_stat
        return nll, grad

    lo, hi = theta_bounds
    bounds = [(lo, hi)] * p

    opt = minimize(
        fun=lambda th: objective(th)[0],
        x0=theta_init,
        jac=lambda th: objective(th)[1],
        method="L-BFGS-B",
        bounds=bounds,
    )

    theta_hat = opt.x
    eta_hat = S @ theta_hat
    logZ_hat = logsumexp(eta_hat)
    probs_hat = np.exp(eta_hat - logZ_hat)

    return {
        "theta_hat": theta_hat,
        "y_hat": np.exp(theta_hat),
        "logZ_hat": float(logZ_hat),
        "loglik_hat": float(sufficient_stat @ theta_hat - n * logZ_hat),
        "probs_hat": probs_hat,
        "expected_state_counts": n * probs_hat,
        "fitted_marginals": probs_hat @ S,
        "optimizer_success": bool(opt.success),
        "optimizer_message": str(opt.message),
    }


def deviance_statistic(counts, probs):
    counts = np.asarray(counts, dtype=float)
    probs = np.asarray(probs, dtype=float)

    n = counts.sum()
    expected = n * probs

    mask = counts > 0
    return float(2.0 * np.sum(counts[mask] * np.log(counts[mask] / expected[mask])))


# ============================================================
# TEST: WYBIERAMY OBSERWACJE Z NOŚNIKA I TESTUJEMY multG(1,y)
# ============================================================

def test_filtered_multG1_star_from_csv(
    csv_path,
    bootstrap_B=300,
    refit_in_bootstrap=True,
    seed=123,
):
    X = read_experiment_data(csv_path)
    n_total = X.shape[0]

    is_valid = valid_rows_mask(X, EDGES)
    X_valid = X[is_valid]
    n_valid = X_valid.shape[0]
    n_invalid = n_total - n_valid

    if n_valid == 0:
        raise ValueError("Po odrzuceniu obserwacji spoza nośnika nie zostały żadne dane.")

    support_masks = enumerate_independent_set_masks(P, EDGES)
    support_states = masks_to_matrix(support_masks, P)
    mask_to_index = {int(m): i for i, m in enumerate(support_masks.tolist())}

    valid_masks = matrix_to_masks(X_valid)
    obs_counts = np.zeros(len(support_masks), dtype=int)
    for m in valid_masks:
        obs_counts[mask_to_index[int(m)]] += 1

    fit = fit_multG1_from_state_counts(
        state_counts=obs_counts,
        support_states=support_states,
    )

    D_obs = deviance_statistic(obs_counts, fit["probs_hat"])

    rng = np.random.default_rng(seed)
    D_boot = np.empty(bootstrap_B, dtype=float)

    for b in range(bootstrap_B):
        boot_counts = rng.multinomial(n_valid, fit["probs_hat"])

        if refit_in_bootstrap:
            fit_b = fit_multG1_from_state_counts(
                state_counts=boot_counts,
                support_states=support_states,
                theta_init=fit["theta_hat"],
            )
            D_boot[b] = deviance_statistic(boot_counts, fit_b["probs_hat"])
        else:
            D_boot[b] = deviance_statistic(boot_counts, fit["probs_hat"])

    p_value = (1.0 + np.sum(D_boot >= D_obs)) / (bootstrap_B + 1.0)

    state_table = pd.DataFrame({
        "state_mask": support_masks,
        "state": [mask_to_bitstring(int(m), P) for m in support_masks],
        "obs": obs_counts,
        "exp": fit["expected_state_counts"],
        "prob_hat": fit["probs_hat"],
    }).sort_values(["obs", "exp"], ascending=[False, False]).reset_index(drop=True)

    marginal_table = pd.DataFrame({
        "vertex": ["1", "2", "3", "4", "5", "6", "7"],
        "obs_mean_filtered": X_valid.mean(axis=0),
        "fit_mean": fit["fitted_marginals"],
        "diff": X_valid.mean(axis=0) - fit["fitted_marginals"],
        "y_hat": fit["y_hat"],
        "theta_hat": fit["theta_hat"],
    })

    return {
        "summary": {
            "csv_path": csv_path,
            "graph": "star: edges (1,2), (1,3), (1,4), (1,5), (1,6), (1,7)",
            "edges_python": EDGES,
            "n_total": int(n_total),
            "n_valid": int(n_valid),
            "n_invalid": int(n_invalid),
            "frac_valid": float(n_valid / n_total),
            "support_size": int(len(support_masks)),
            "deviance_obs": float(D_obs),
            "bootstrap_p_value": float(p_value),
            "reject_multG_at_05": bool(p_value < 0.05),
            "optimizer_success": bool(fit["optimizer_success"]),
            "optimizer_message": fit["optimizer_message"],
        },
        "fit": fit,
        "state_table": state_table,
        "marginal_table": marginal_table,
        "X_valid": X_valid,
        "valid_rows_mask": is_valid,
        "support_states": support_states,
        "support_masks": support_masks,
        "obs_counts": obs_counts,
        "D_boot": D_boot,
    }


def print_report_star(result, top_k=20, digits=4):
    s = result["summary"]

    print("=== TEST multG(1,y) PO OGRANICZENIU DO NOŚNIKA ===")
    print(f"csv_path             = {s['csv_path']}")
    print(f"graph                = {s['graph']}")
    print(f"edges_python         = {s['edges_python']}")
    print(f"n_total              = {s['n_total']}")
    print(f"n_valid              = {s['n_valid']}")
    print(f"n_invalid            = {s['n_invalid']}")
    print(f"frac_valid           = {s['frac_valid']:.4f}")
    print(f"support_size         = {s['support_size']}")
    print(f"deviance_obs         = {s['deviance_obs']:.4f}")
    print(f"bootstrap_p_value    = {s['bootstrap_p_value']:.4f}")
    print(f"reject_multG_at_05   = {s['reject_multG_at_05']}")
    print(f"optimizer_success    = {s['optimizer_success']}")
    print(f"optimizer_message    = {s['optimizer_message']}")
    print()

    print("=== MARGINALS DLA DANYCH Z NOŚNIKA ===")
    print(result["marginal_table"].round(digits).to_string(index=False))
    print()

    print(f"=== TOP {top_k} STANÓW Z NOŚNIKA ===")
    print(result["state_table"].head(top_k).round(digits).to_string(index=False))


# ============================================================
# URUCHOMIENIE
# ============================================================

result = test_filtered_multG1_star_from_csv(
    csv_path=CSV_PATH,
    bootstrap_B=BOOTSTRAP_B,
    refit_in_bootstrap=REFIT_IN_BOOTSTRAP,
    seed=SEED,
)

print_report_star(result, top_k=20, digits=4)

=== TEST multG(1,y) PO OGRANICZENIU DO NOŚNIKA ===
csv_path             = experiment_data_fig4d.csv
graph                = star: edges (1,2), (1,3), (1,4), (1,5), (1,6), (1,7)
edges_python         = [(0, 1), (0, 2), (0, 3), (0, 4), (0, 5), (0, 6)]
n_total              = 1033
n_valid              = 268
n_invalid            = 765
frac_valid           = 0.2594
support_size         = 65
deviance_obs         = 106.8441
bootstrap_p_value    = 0.0040
reject_multG_at_05   = True
optimizer_success    = True
optimizer_message    = CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*EPSMCH

=== MARGINALS DLA DANYCH Z NOŚNIKA ===
vertex  obs_mean_filtered  fit_mean  diff   y_hat  theta_hat
     1             0.3955    0.3955  -0.0 17.6210     2.8691
     2             0.2985    0.2985   0.0  0.9756    -0.0247
     3             0.2463    0.2463  -0.0  0.6875    -0.3747
     4             0.2015    0.2015  -0.0  0.5000    -0.6931
     5             0.2836    0.2836  -0.0  0.8837    -0.1236
     6        

In [16]:
import numpy as np
import pandas as pd
from scipy.optimize import minimize
from scipy.special import logsumexp


# ============================================================
# USTAWIENIA
# ============================================================

CSV_PATH = "experiment_data_fig2e.csv"

# Graf z obrazka (b): cykl C4
# 1--2--3--4--1
EDGES = [
    (0, 1),  # 1--2
    (1, 2),  # 2--3
    (2, 3),  # 3--4
    (0, 3),  # 1--4
]

P = 4
BOOTSTRAP_B = 500
REFIT_IN_BOOTSTRAP = True
SEED = 123


# ============================================================
# WCZYTANIE DANYCH
# ============================================================

def read_experiment_data(csv_path):
    """
    Plik:
      - co najmniej 4 wiersze = wierzchołki
      - kolumny = obserwacje

    Bierzemy pierwsze 4 wiersze i transponujemy.
    """
    df = pd.read_csv(csv_path, header=None)
    df = df.dropna(axis=0, how="all").dropna(axis=1, how="all")

    if df.shape[0] < 4:
        raise ValueError(
            f"Oczekiwałem co najmniej 4 wierszy w pliku {csv_path}, a jest {df.shape[0]}."
        )

    X_raw = df.iloc[:4, :].to_numpy()

    uniq = np.unique(X_raw)
    if not np.all(np.isin(uniq, [0, 1])):
        raise ValueError(f"Dane muszą być binarne 0/1. Unikalne wartości: {uniq}")

    return X_raw.T.astype(int)


# ============================================================
# NARZĘDZIA
# ============================================================

def valid_rows_mask(X, edges):
    ok = np.ones(X.shape[0], dtype=bool)
    for u, v in edges:
        ok &= ~((X[:, u] == 1) & (X[:, v] == 1))
    return ok


def enumerate_independent_set_masks(p, edges):
    masks = []
    for mask in range(1 << p):
        good = True
        for u, v in edges:
            if ((mask >> u) & 1) and ((mask >> v) & 1):
                good = False
                break
        if good:
            masks.append(mask)
    return np.array(masks, dtype=np.int64)


def masks_to_matrix(masks, p):
    masks = np.asarray(masks, dtype=np.int64)
    return ((masks[:, None] >> np.arange(p, dtype=np.int64)) & 1).astype(int)


def matrix_to_masks(X):
    p = X.shape[1]
    powers = (1 << np.arange(p, dtype=np.int64))
    return (X.astype(np.int64) * powers).sum(axis=1)


def mask_to_bitstring(mask, p):
    return "".join(str((mask >> j) & 1) for j in range(p))


# ============================================================
# DOPASOWANIE multG(1,y)
# ============================================================

def fit_multG1_from_state_counts(
    state_counts,
    support_states,
    theta_init=None,
    theta_bounds=(-20.0, 20.0),
):
    counts = np.asarray(state_counts, dtype=float)
    S = np.asarray(support_states, dtype=float)

    n = counts.sum()
    if n <= 0:
        raise ValueError("Brak obserwacji w nośniku.")

    sufficient_stat = counts @ S
    p = S.shape[1]

    if theta_init is None:
        theta_init = np.zeros(p, dtype=float)
    else:
        theta_init = np.asarray(theta_init, dtype=float)

    def objective(theta):
        eta = S @ theta
        logZ = logsumexp(eta)
        probs = np.exp(eta - logZ)

        nll = n * logZ - sufficient_stat @ theta
        grad = n * (probs @ S) - sufficient_stat
        return nll, grad

    lo, hi = theta_bounds
    bounds = [(lo, hi)] * p

    opt = minimize(
        fun=lambda th: objective(th)[0],
        x0=theta_init,
        jac=lambda th: objective(th)[1],
        method="L-BFGS-B",
        bounds=bounds,
    )

    theta_hat = opt.x
    eta_hat = S @ theta_hat
    logZ_hat = logsumexp(eta_hat)
    probs_hat = np.exp(eta_hat - logZ_hat)

    return {
        "theta_hat": theta_hat,
        "y_hat": np.exp(theta_hat),
        "logZ_hat": float(logZ_hat),
        "loglik_hat": float(sufficient_stat @ theta_hat - n * logZ_hat),
        "probs_hat": probs_hat,
        "expected_state_counts": n * probs_hat,
        "fitted_marginals": probs_hat @ S,
        "optimizer_success": bool(opt.success),
        "optimizer_message": str(opt.message),
    }


def deviance_statistic(counts, probs):
    counts = np.asarray(counts, dtype=float)
    probs = np.asarray(probs, dtype=float)

    n = counts.sum()
    expected = n * probs

    mask = counts > 0
    return float(2.0 * np.sum(counts[mask] * np.log(counts[mask] / expected[mask])))


# ============================================================
# TEST: WYBIERAMY OBSERWACJE Z NOŚNIKA I TESTUJEMY multG(1,y)
# ============================================================

def test_filtered_multG1_C4_from_csv(
    csv_path,
    bootstrap_B=300,
    refit_in_bootstrap=True,
    seed=123,
):
    X = read_experiment_data(csv_path)
    n_total = X.shape[0]

    is_valid = valid_rows_mask(X, EDGES)
    X_valid = X[is_valid]
    n_valid = X_valid.shape[0]
    n_invalid = n_total - n_valid

    if n_valid == 0:
        raise ValueError("Po odrzuceniu obserwacji spoza nośnika nie zostały żadne dane.")

    support_masks = enumerate_independent_set_masks(P, EDGES)
    support_states = masks_to_matrix(support_masks, P)
    mask_to_index = {int(m): i for i, m in enumerate(support_masks.tolist())}

    valid_masks = matrix_to_masks(X_valid)
    obs_counts = np.zeros(len(support_masks), dtype=int)
    for m in valid_masks:
        obs_counts[mask_to_index[int(m)]] += 1

    fit = fit_multG1_from_state_counts(
        state_counts=obs_counts,
        support_states=support_states,
    )

    D_obs = deviance_statistic(obs_counts, fit["probs_hat"])

    rng = np.random.default_rng(seed)
    D_boot = np.empty(bootstrap_B, dtype=float)

    for b in range(bootstrap_B):
        boot_counts = rng.multinomial(n_valid, fit["probs_hat"])

        if refit_in_bootstrap:
            fit_b = fit_multG1_from_state_counts(
                state_counts=boot_counts,
                support_states=support_states,
                theta_init=fit["theta_hat"],
            )
            D_boot[b] = deviance_statistic(boot_counts, fit_b["probs_hat"])
        else:
            D_boot[b] = deviance_statistic(boot_counts, fit["probs_hat"])

    p_value = (1.0 + np.sum(D_boot >= D_obs)) / (bootstrap_B + 1.0)

    state_table = pd.DataFrame({
        "state_mask": support_masks,
        "state": [mask_to_bitstring(int(m), P) for m in support_masks],
        "obs": obs_counts,
        "exp": fit["expected_state_counts"],
        "prob_hat": fit["probs_hat"],
    }).sort_values(["obs", "exp"], ascending=[False, False]).reset_index(drop=True)

    marginal_table = pd.DataFrame({
        "vertex": ["1", "2", "3", "4"],
        "obs_mean_filtered": X_valid.mean(axis=0),
        "fit_mean": fit["fitted_marginals"],
        "diff": X_valid.mean(axis=0) - fit["fitted_marginals"],
        "y_hat": fit["y_hat"],
        "theta_hat": fit["theta_hat"],
    })

    return {
        "summary": {
            "csv_path": csv_path,
            "graph": "C4: edges (1,2), (2,3), (3,4), (4,1)",
            "edges_python": EDGES,
            "n_total": int(n_total),
            "n_valid": int(n_valid),
            "n_invalid": int(n_invalid),
            "frac_valid": float(n_valid / n_total),
            "support_size": int(len(support_masks)),
            "deviance_obs": float(D_obs),
            "bootstrap_p_value": float(p_value),
            "reject_multG_at_05": bool(p_value < 0.05),
            "optimizer_success": bool(fit["optimizer_success"]),
            "optimizer_message": fit["optimizer_message"],
        },
        "fit": fit,
        "state_table": state_table,
        "marginal_table": marginal_table,
        "X_valid": X_valid,
        "valid_rows_mask": is_valid,
        "support_states": support_states,
        "support_masks": support_masks,
        "obs_counts": obs_counts,
        "D_boot": D_boot,
    }


def print_report_C4(result, top_k=20, digits=4):
    s = result["summary"]

    print("=== TEST multG(1,y) PO OGRANICZENIU DO NOŚNIKA ===")
    print(f"csv_path             = {s['csv_path']}")
    print(f"graph                = {s['graph']}")
    print(f"edges_python         = {s['edges_python']}")
    print(f"n_total              = {s['n_total']}")
    print(f"n_valid              = {s['n_valid']}")
    print(f"n_invalid            = {s['n_invalid']}")
    print(f"frac_valid           = {s['frac_valid']:.4f}")
    print(f"support_size         = {s['support_size']}")
    print(f"deviance_obs         = {s['deviance_obs']:.4f}")
    print(f"bootstrap_p_value    = {s['bootstrap_p_value']:.4f}")
    print(f"reject_multG_at_05   = {s['reject_multG_at_05']}")
    print(f"optimizer_success    = {s['optimizer_success']}")
    print(f"optimizer_message    = {s['optimizer_message']}")
    print()

    print("=== MARGINALS DLA DANYCH Z NOŚNIKA ===")
    print(result["marginal_table"].round(digits).to_string(index=False))
    print()

    print(f"=== TOP {top_k} STANÓW Z NOŚNIKA ===")
    print(result["state_table"].head(top_k).round(digits).to_string(index=False))


# ============================================================
# URUCHOMIENIE
# ============================================================

result = test_filtered_multG1_C4_from_csv(
    csv_path=CSV_PATH,
    bootstrap_B=BOOTSTRAP_B,
    refit_in_bootstrap=REFIT_IN_BOOTSTRAP,
    seed=SEED,
)

print_report_C4(result, top_k=20, digits=4)

=== TEST multG(1,y) PO OGRANICZENIU DO NOŚNIKA ===
csv_path             = experiment_data_fig2e.csv
graph                = C4: edges (1,2), (2,3), (3,4), (4,1)
edges_python         = [(0, 1), (1, 2), (2, 3), (0, 3)]
n_total              = 672
n_valid              = 261
n_invalid            = 411
frac_valid           = 0.3884
support_size         = 7
deviance_obs         = 2.7639
bootstrap_p_value    = 0.2275
reject_multG_at_05   = False
optimizer_success    = True
optimizer_message    = CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*EPSMCH

=== MARGINALS DLA DANYCH Z NOŚNIKA ===
vertex  obs_mean_filtered  fit_mean  diff   y_hat  theta_hat
     1             0.4904    0.4904  -0.0 36.8408     3.6066
     2             0.4674    0.4674   0.0 15.7429     2.7564
     3             0.4713    0.4713  -0.0 14.5144     2.6751
     4             0.4828    0.4828   0.0 33.6042     3.5146

=== TOP 20 STANÓW Z NOŚNIKA ===
 state_mask state  obs      exp  prob_hat
         10  0101  120 118.4744    

# Korekta

In [12]:
import numpy as np
import pandas as pd
from scipy.optimize import minimize
from scipy.special import logsumexp


# ============================================================
# USTAWIENIA
# ============================================================

CSV_PATH = "experiment_data_fig2d.csv"

# Gwiazda na 4 wierzchołkach:
# 1--2, 2--3, 3--4
EDGES = [
    (0, 1),
    (1, 2),
    (2, 3),
]

P = 4
BOOTSTRAP_B = 500
REFIT_IN_BOOTSTRAP = True
SEED = 123

# alpha_i = P(obs=1 | true=0)
# beta_i  = P(obs=0 | true=1)
ALPHA = np.array([0.18, 0.18, 0.18, 0.18], dtype=float)
BETA  = np.array([0.03, 0.03, 0.03, 0.03], dtype=float)


# ============================================================
# WCZYTANIE DANYCH
# ============================================================

def read_experiment_data(csv_path):
    df = pd.read_csv(csv_path, header=None)
    df = df.dropna(axis=0, how="all").dropna(axis=1, how="all")

    if df.shape[0] < 4:
        raise ValueError(
            f"Oczekiwałem co najmniej 4 wierszy w pliku {csv_path}, a jest {df.shape[0]}."
        )

    X_raw = df.iloc[:4, :].to_numpy()

    uniq = np.unique(X_raw)
    if not np.all(np.isin(uniq, [0, 1])):
        raise ValueError(f"Dane muszą być binarne 0/1. Unikalne wartości: {uniq}")

    return X_raw.T.astype(int)


# ============================================================
# KOREKTA BITOWA
# ============================================================

def estimate_true_marginals_from_readout(obs_mean, alpha, beta):
    """
    Z równania:
        E[X_obs] = alpha + (1-alpha-beta) * p_true
    """
    denom = 1.0 - alpha - beta
    if np.any(denom <= 0):
        raise ValueError("Dla każdego wierzchołka musi być 1 - alpha - beta > 0.")

    p_true = (obs_mean - alpha) / denom
    return np.clip(p_true, 1e-6, 1 - 1e-6)


def bitwise_bayes_correction(X_obs, alpha, beta, prior_true):
    """
    Dla każdego bitu osobno:
      - liczymy P(true=1 | obs)
      - ustawiamy corrected=1 wtedy i tylko wtedy, gdy posterior > 0.5
    """
    X_obs = np.asarray(X_obs, dtype=int)
    n, p = X_obs.shape
    X_corr = np.zeros_like(X_obs)

    for j in range(p):
        pi = prior_true[j]
        a = alpha[j]
        b = beta[j]

        # posterior gdy obs=1
        post1 = ((1 - b) * pi) / (((1 - b) * pi) + a * (1 - pi))

        # posterior gdy obs=0
        post0 = (b * pi) / ((b * pi) + (1 - a) * (1 - pi))

        X_corr[X_obs[:, j] == 1, j] = int(post1 > 0.5)
        X_corr[X_obs[:, j] == 0, j] = int(post0 > 0.5)

    return X_corr


# ============================================================
# NARZĘDZIA
# ============================================================

def valid_rows_mask(X, edges):
    ok = np.ones(X.shape[0], dtype=bool)
    for u, v in edges:
        ok &= ~((X[:, u] == 1) & (X[:, v] == 1))
    return ok


def enumerate_independent_set_masks(p, edges):
    masks = []
    for mask in range(1 << p):
        good = True
        for u, v in edges:
            if ((mask >> u) & 1) and ((mask >> v) & 1):
                good = False
                break
        if good:
            masks.append(mask)
    return np.array(masks, dtype=np.int64)


def masks_to_matrix(masks, p):
    masks = np.asarray(masks, dtype=np.int64)
    return ((masks[:, None] >> np.arange(p, dtype=np.int64)) & 1).astype(int)


def matrix_to_masks(X):
    p = X.shape[1]
    powers = (1 << np.arange(p, dtype=np.int64))
    return (X.astype(np.int64) * powers).sum(axis=1)


def mask_to_bitstring(mask, p):
    return "".join(str((mask >> j) & 1) for j in range(p))


# ============================================================
# DOPASOWANIE multG(1,y)
# ============================================================

def fit_multG1_from_state_counts(
    state_counts,
    support_states,
    theta_init=None,
    theta_bounds=(-20.0, 20.0),
):
    counts = np.asarray(state_counts, dtype=float)
    S = np.asarray(support_states, dtype=float)

    n = counts.sum()
    if n <= 0:
        raise ValueError("Brak obserwacji w nośniku.")

    sufficient_stat = counts @ S
    p = S.shape[1]

    if theta_init is None:
        theta_init = np.zeros(p, dtype=float)
    else:
        theta_init = np.asarray(theta_init, dtype=float)

    def objective(theta):
        eta = S @ theta
        logZ = logsumexp(eta)
        probs = np.exp(eta - logZ)

        nll = n * logZ - sufficient_stat @ theta
        grad = n * (probs @ S) - sufficient_stat
        return nll, grad

    lo, hi = theta_bounds
    bounds = [(lo, hi)] * p

    opt = minimize(
        fun=lambda th: objective(th)[0],
        x0=theta_init,
        jac=lambda th: objective(th)[1],
        method="L-BFGS-B",
        bounds=bounds,
    )

    theta_hat = opt.x
    eta_hat = S @ theta_hat
    logZ_hat = logsumexp(eta_hat)
    probs_hat = np.exp(eta_hat - logZ_hat)

    return {
        "theta_hat": theta_hat,
        "y_hat": np.exp(theta_hat),
        "logZ_hat": float(logZ_hat),
        "loglik_hat": float(sufficient_stat @ theta_hat - n * logZ_hat),
        "probs_hat": probs_hat,
        "expected_state_counts": n * probs_hat,
        "fitted_marginals": probs_hat @ S,
        "optimizer_success": bool(opt.success),
        "optimizer_message": str(opt.message),
    }


def deviance_statistic(counts, probs):
    counts = np.asarray(counts, dtype=float)
    probs = np.asarray(probs, dtype=float)

    n = counts.sum()
    expected = n * probs
    expected = np.clip(expected, 1e-300, None)

    mask = counts > 0
    return float(2.0 * np.sum(counts[mask] * np.log(counts[mask] / expected[mask])))


# ============================================================
# GŁÓWNY TEST:
# KOREKTA -> FILTRACJA DO NOŚNIKA -> TEST multG
# ============================================================

def test_corrected_then_filtered_multG1_star4(
    csv_path,
    alpha,
    beta,
    bootstrap_B=300,
    refit_in_bootstrap=True,
    seed=123,
):
    X_obs = read_experiment_data(csv_path)
    n_total = X_obs.shape[0]

    # 1. estymacja marginesów true z prostego modelu readout
    obs_mean = X_obs.mean(axis=0)
    prior_true = estimate_true_marginals_from_readout(obs_mean, alpha, beta)

    # 2. korekta bitowa
    X_corr = bitwise_bayes_correction(X_obs, alpha, beta, prior_true)

    # 3. usuwanie stanów spoza nośnika po korekcie
    is_valid = valid_rows_mask(X_corr, EDGES)
    X_valid = X_corr[is_valid]
    n_valid = X_valid.shape[0]
    n_invalid = n_total - n_valid

    if n_valid == 0:
        raise ValueError("Po korekcie i filtracji do nośnika nie zostały żadne dane.")

    # 4. nośnik modelu
    support_masks = enumerate_independent_set_masks(P, EDGES)
    support_states = masks_to_matrix(support_masks, P)
    mask_to_index = {int(m): i for i, m in enumerate(support_masks.tolist())}

    # 5. liczniki stanów
    valid_masks = matrix_to_masks(X_valid)
    obs_counts = np.zeros(len(support_masks), dtype=int)
    for m in valid_masks:
        obs_counts[mask_to_index[int(m)]] += 1

    # 6. dopasowanie multG(1,y)
    fit = fit_multG1_from_state_counts(
        state_counts=obs_counts,
        support_states=support_states,
    )

    D_obs = deviance_statistic(obs_counts, fit["probs_hat"])

    # 7. bootstrap
    rng = np.random.default_rng(seed)
    D_boot = np.empty(bootstrap_B, dtype=float)

    for b in range(bootstrap_B):
        boot_counts = rng.multinomial(n_valid, fit["probs_hat"])

        if refit_in_bootstrap:
            fit_b = fit_multG1_from_state_counts(
                state_counts=boot_counts,
                support_states=support_states,
                theta_init=fit["theta_hat"],
            )
            D_boot[b] = deviance_statistic(boot_counts, fit_b["probs_hat"])
        else:
            D_boot[b] = deviance_statistic(boot_counts, fit["probs_hat"])

    p_value = (1.0 + np.sum(D_boot >= D_obs)) / (bootstrap_B + 1.0)

    state_table = pd.DataFrame({
        "state_mask": support_masks,
        "state": [mask_to_bitstring(int(m), P) for m in support_masks],
        "obs": obs_counts,
        "exp": fit["expected_state_counts"],
        "prob_hat": fit["probs_hat"],
    }).sort_values(["obs", "exp"], ascending=[False, False]).reset_index(drop=True)

    marginal_table = pd.DataFrame({
        "vertex": ["1", "2", "3", "4"],
        "obs_mean_raw": X_obs.mean(axis=0),
        "corr_mean_valid_only": X_valid.mean(axis=0),
        "fit_mean": fit["fitted_marginals"],
        "diff": X_valid.mean(axis=0) - fit["fitted_marginals"],
        "y_hat": fit["y_hat"],
        "theta_hat": fit["theta_hat"],
        "alpha": alpha,
        "beta": beta,
        "prior_true_used_in_correction": prior_true,
    })

    return {
        "summary": {
            "csv_path": csv_path,
            "graph": "star4: edges (1,2), (1,3), (1,4)",
            "edges_python": EDGES,
            "n_total": int(n_total),
            "n_valid_after_correction_and_filter": int(n_valid),
            "n_invalid_after_correction_and_filter": int(n_invalid),
            "frac_valid_after_correction_and_filter": float(n_valid / n_total),
            "support_size": int(len(support_masks)),
            "deviance_obs": float(D_obs),
            "bootstrap_p_value": float(p_value),
            "reject_multG_at_05": bool(p_value < 0.05),
            "optimizer_success": bool(fit["optimizer_success"]),
            "optimizer_message": fit["optimizer_message"],
        },
        "fit": fit,
        "marginal_table": marginal_table,
        "state_table": state_table,
        "X_obs": X_obs,
        "X_corr": X_corr,
        "X_valid": X_valid,
        "valid_rows_mask_after_correction": is_valid,
        "support_states": support_states,
        "support_masks": support_masks,
        "obs_counts": obs_counts,
        "D_boot": D_boot,
    }


def print_report_corrected_then_filtered(result, top_k=20, digits=4):
    s = result["summary"]

    print("=== TEST multG(1,y): KOREKTA -> FILTRACJA DO NOŚNIKA -> TEST ===")
    print(f"csv_path                              = {s['csv_path']}")
    print(f"graph                                 = {s['graph']}")
    print(f"edges_python                          = {s['edges_python']}")
    print(f"n_total                               = {s['n_total']}")
    print(f"n_valid_after_correction_and_filter   = {s['n_valid_after_correction_and_filter']}")
    print(f"n_invalid_after_correction_and_filter = {s['n_invalid_after_correction_and_filter']}")
    print(f"frac_valid_after_correction_and_filter= {s['frac_valid_after_correction_and_filter']:.4f}")
    print(f"support_size                          = {s['support_size']}")
    print(f"deviance_obs                          = {s['deviance_obs']:.4f}")
    print(f"bootstrap_p_value                     = {s['bootstrap_p_value']:.4f}")
    print(f"reject_multG_at_05                    = {s['reject_multG_at_05']}")
    print(f"optimizer_success                     = {s['optimizer_success']}")
    print(f"optimizer_message                     = {s['optimizer_message']}")
    print()

    print("=== MARGINALS ===")
    print(result["marginal_table"].round(digits).to_string(index=False))
    print()

    print(f"=== TOP {top_k} STANÓW Z NOŚNIKA PO KOREKCIE ===")
    print(result["state_table"].head(top_k).round(digits).to_string(index=False))


# ============================================================
# URUCHOMIENIE
# ============================================================

result = test_corrected_then_filtered_multG1_star4(
    csv_path=CSV_PATH,
    alpha=ALPHA,
    beta=BETA,
    bootstrap_B=BOOTSTRAP_B,
    refit_in_bootstrap=REFIT_IN_BOOTSTRAP,
    seed=SEED,
)

print_report_corrected_then_filtered(result, top_k=20, digits=4)

=== TEST multG(1,y): KOREKTA -> FILTRACJA DO NOŚNIKA -> TEST ===
csv_path                              = experiment_data_fig2d.csv
graph                                 = star4: edges (1,2), (1,3), (1,4)
edges_python                          = [(0, 1), (1, 2), (2, 3)]
n_total                               = 678
n_valid_after_correction_and_filter   = 191
n_invalid_after_correction_and_filter = 487
frac_valid_after_correction_and_filter= 0.2817
support_size                          = 8
deviance_obs                          = 0.5081
bootstrap_p_value                     = 0.9102
reject_multG_at_05                    = False
optimizer_success                     = True
optimizer_message                     = CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*EPSMCH

=== MARGINALS ===
vertex  obs_mean_raw  corr_mean_valid_only  fit_mean  diff   y_hat  theta_hat  alpha  beta  prior_true_used_in_correction
     1        0.5265                0.4398    0.4398  -0.0  3.1111     1.1350   0.18  0.03 

In [14]:
import numpy as np
import pandas as pd
from scipy.optimize import minimize
from scipy.special import logsumexp


# ============================================================
# USTAWIENIA
# ============================================================

CSV_PATH = "experiment_data_fig2d.csv"

# Ścieżka na 4 wierzchołkach: 1--2--3--4
EDGES = [
    (0, 1),
    (1, 2),
    (2, 3),
]

P = 4
BOOTSTRAP_B = 500
REFIT_IN_BOOTSTRAP = True
SEED = 123

# alpha_i = P(obs=1 | true=0)
# beta_i  = P(obs=0 | true=1)
ALPHA = np.array([0.18, 0.18, 0.18, 0.18], dtype=float)
BETA  = np.array([0.03, 0.03, 0.03, 0.03], dtype=float)


# ============================================================
# WCZYTANIE DANYCH
# ============================================================

def read_experiment_data(csv_path):
    df = pd.read_csv(csv_path, header=None)
    df = df.dropna(axis=0, how="all").dropna(axis=1, how="all")

    if df.shape[0] < 4:
        raise ValueError(
            f"Oczekiwałem co najmniej 4 wierszy w pliku {csv_path}, a jest {df.shape[0]}."
        )

    X_raw = df.iloc[:4, :].to_numpy()

    uniq = np.unique(X_raw)
    if not np.all(np.isin(uniq, [0, 1])):
        raise ValueError(f"Dane muszą być binarne 0/1. Unikalne wartości: {uniq}")

    return X_raw.T.astype(int)


# ============================================================
# KOREKTA BITOWA
# ============================================================

def estimate_true_marginals_from_readout(obs_mean, alpha, beta):
    """
    Z równania:
        E[X_obs] = alpha + (1-alpha-beta) * p_true
    """
    denom = 1.0 - alpha - beta
    if np.any(denom <= 0):
        raise ValueError("Dla każdego wierzchołka musi być 1 - alpha - beta > 0.")

    p_true = (obs_mean - alpha) / denom
    return np.clip(p_true, 1e-6, 1 - 1e-6)


def bitwise_bayes_correction(X_obs, alpha, beta, prior_true):
    """
    Dla każdego bitu osobno:
      - liczymy P(true=1 | obs)
      - ustawiamy corrected=1 wtedy i tylko wtedy, gdy posterior > 0.5
    """
    X_obs = np.asarray(X_obs, dtype=int)
    n, p = X_obs.shape
    X_corr = np.zeros_like(X_obs)

    for j in range(p):
        pi = prior_true[j]
        a = alpha[j]
        b = beta[j]

        # posterior gdy obs=1
        post1 = ((1 - b) * pi) / (((1 - b) * pi) + a * (1 - pi))

        # posterior gdy obs=0
        post0 = (b * pi) / ((b * pi) + (1 - a) * (1 - pi))

        X_corr[X_obs[:, j] == 1, j] = int(post1 > 0.5)
        X_corr[X_obs[:, j] == 0, j] = int(post0 > 0.5)

    return X_corr


# ============================================================
# NARZĘDZIA
# ============================================================

def valid_rows_mask(X, edges):
    ok = np.ones(X.shape[0], dtype=bool)
    for u, v in edges:
        ok &= ~((X[:, u] == 1) & (X[:, v] == 1))
    return ok


def enumerate_independent_set_masks(p, edges):
    masks = []
    for mask in range(1 << p):
        good = True
        for u, v in edges:
            if ((mask >> u) & 1) and ((mask >> v) & 1):
                good = False
                break
        if good:
            masks.append(mask)
    return np.array(masks, dtype=np.int64)


def masks_to_matrix(masks, p):
    masks = np.asarray(masks, dtype=np.int64)
    return ((masks[:, None] >> np.arange(p, dtype=np.int64)) & 1).astype(int)


def matrix_to_masks(X):
    p = X.shape[1]
    powers = (1 << np.arange(p, dtype=np.int64))
    return (X.astype(np.int64) * powers).sum(axis=1)


def mask_to_bitstring(mask, p):
    return "".join(str((mask >> j) & 1) for j in range(p))


# ============================================================
# DOPASOWANIE multG(1,y)
# ============================================================

def fit_multG1_from_state_counts(
    state_counts,
    support_states,
    theta_init=None,
    theta_bounds=(-20.0, 20.0),
):
    counts = np.asarray(state_counts, dtype=float)
    S = np.asarray(support_states, dtype=float)

    n = counts.sum()
    if n <= 0:
        raise ValueError("Brak obserwacji w nośniku.")

    sufficient_stat = counts @ S
    p = S.shape[1]

    if theta_init is None:
        theta_init = np.zeros(p, dtype=float)
    else:
        theta_init = np.asarray(theta_init, dtype=float)

    def objective(theta):
        eta = S @ theta
        logZ = logsumexp(eta)
        probs = np.exp(eta - logZ)

        nll = n * logZ - sufficient_stat @ theta
        grad = n * (probs @ S) - sufficient_stat
        return nll, grad

    lo, hi = theta_bounds
    bounds = [(lo, hi)] * p

    opt = minimize(
        fun=lambda th: objective(th)[0],
        x0=theta_init,
        jac=lambda th: objective(th)[1],
        method="L-BFGS-B",
        bounds=bounds,
    )

    theta_hat = opt.x
    eta_hat = S @ theta_hat
    logZ_hat = logsumexp(eta_hat)
    probs_hat = np.exp(eta_hat - logZ_hat)

    return {
        "theta_hat": theta_hat,
        "y_hat": np.exp(theta_hat),
        "logZ_hat": float(logZ_hat),
        "loglik_hat": float(sufficient_stat @ theta_hat - n * logZ_hat),
        "probs_hat": probs_hat,
        "expected_state_counts": n * probs_hat,
        "fitted_marginals": probs_hat @ S,
        "optimizer_success": bool(opt.success),
        "optimizer_message": str(opt.message),
    }


def deviance_statistic(counts, probs):
    counts = np.asarray(counts, dtype=float)
    probs = np.asarray(probs, dtype=float)

    n = counts.sum()
    expected = n * probs
    expected = np.clip(expected, 1e-300, None)

    mask = counts > 0
    return float(2.0 * np.sum(counts[mask] * np.log(counts[mask] / expected[mask])))


# ============================================================
# GŁÓWNY TEST:
# KORYGUJ TYLKO OBSERWACJE SPOZA NOŚNIKA
# ============================================================

def test_correct_only_invalid_then_filter_multG1(
    csv_path,
    alpha,
    beta,
    bootstrap_B=300,
    refit_in_bootstrap=True,
    seed=123,
):
    X_obs = read_experiment_data(csv_path)
    n_total = X_obs.shape[0]

    # 1. które obserwacje pierwotnie są w nośniku?
    is_valid_raw = valid_rows_mask(X_obs, EDGES)
    n_valid_raw = int(is_valid_raw.sum())
    n_invalid_raw = int((~is_valid_raw).sum())

    # 2. estymacja marginesów true z prostego modelu readout
    obs_mean = X_obs.mean(axis=0)
    prior_true = estimate_true_marginals_from_readout(obs_mean, alpha, beta)

    # 3. korekta bitowa liczona dla wszystkich
    X_corr_all = bitwise_bayes_correction(X_obs, alpha, beta, prior_true)

    # 4. ALE podmieniamy tylko te obserwacje, które były spoza nośnika
    X_changed = X_obs.copy()
    X_changed[~is_valid_raw] = X_corr_all[~is_valid_raw]

    # 5. po tej zmianie znowu sprawdzamy nośnik
    is_valid_after_change = valid_rows_mask(X_changed, EDGES)
    X_valid = X_changed[is_valid_after_change]

    n_valid_after_change = int(is_valid_after_change.sum())
    n_invalid_after_change = int((~is_valid_after_change).sum())
    n_corrected_to_valid = int(np.sum((~is_valid_raw) & is_valid_after_change))
    n_still_invalid = int(np.sum((~is_valid_raw) & (~is_valid_after_change)))

    if n_valid_after_change == 0:
        raise ValueError("Po zmianie obserwacji spoza nośnika nie zostały żadne dane w nośniku.")

    # 6. nośnik modelu
    support_masks = enumerate_independent_set_masks(P, EDGES)
    support_states = masks_to_matrix(support_masks, P)
    mask_to_index = {int(m): i for i, m in enumerate(support_masks.tolist())}

    # 7. liczniki stanów
    valid_masks = matrix_to_masks(X_valid)
    obs_counts = np.zeros(len(support_masks), dtype=int)
    for m in valid_masks:
        obs_counts[mask_to_index[int(m)]] += 1

    # 8. dopasowanie multG(1,y)
    fit = fit_multG1_from_state_counts(
        state_counts=obs_counts,
        support_states=support_states,
    )

    D_obs = deviance_statistic(obs_counts, fit["probs_hat"])

    # 9. bootstrap
    rng = np.random.default_rng(seed)
    D_boot = np.empty(bootstrap_B, dtype=float)

    for b in range(bootstrap_B):
        boot_counts = rng.multinomial(n_valid_after_change, fit["probs_hat"])

        if refit_in_bootstrap:
            fit_b = fit_multG1_from_state_counts(
                state_counts=boot_counts,
                support_states=support_states,
                theta_init=fit["theta_hat"],
            )
            D_boot[b] = deviance_statistic(boot_counts, fit_b["probs_hat"])
        else:
            D_boot[b] = deviance_statistic(boot_counts, fit["probs_hat"])

    p_value = (1.0 + np.sum(D_boot >= D_obs)) / (bootstrap_B + 1.0)

    state_table = pd.DataFrame({
        "state_mask": support_masks,
        "state": [mask_to_bitstring(int(m), P) for m in support_masks],
        "obs": obs_counts,
        "exp": fit["expected_state_counts"],
        "prob_hat": fit["probs_hat"],
    }).sort_values(["obs", "exp"], ascending=[False, False]).reset_index(drop=True)

    marginal_table = pd.DataFrame({
        "vertex": ["1", "2", "3", "4"],
        "obs_mean_raw": X_obs.mean(axis=0),
        "changed_mean_valid_only": X_valid.mean(axis=0),
        "fit_mean": fit["fitted_marginals"],
        "diff": X_valid.mean(axis=0) - fit["fitted_marginals"],
        "y_hat": fit["y_hat"],
        "theta_hat": fit["theta_hat"],
        "alpha": alpha,
        "beta": beta,
        "prior_true_used_in_correction": prior_true,
    })

    return {
        "summary": {
            "csv_path": csv_path,
            "graph": "path4: edges (1,2), (2,3), (3,4)",
            "edges_python": EDGES,
            "n_total": int(n_total),
            "n_valid_raw": n_valid_raw,
            "n_invalid_raw": n_invalid_raw,
            "n_valid_after_change_and_filter": n_valid_after_change,
            "n_invalid_after_change_and_filter": n_invalid_after_change,
            "n_corrected_invalid_to_valid": n_corrected_to_valid,
            "n_still_invalid_after_change": n_still_invalid,
            "frac_valid_after_change_and_filter": float(n_valid_after_change / n_total),
            "support_size": int(len(support_masks)),
            "deviance_obs": float(D_obs),
            "bootstrap_p_value": float(p_value),
            "reject_multG_at_05": bool(p_value < 0.05),
            "optimizer_success": bool(fit["optimizer_success"]),
            "optimizer_message": fit["optimizer_message"],
        },
        "fit": fit,
        "marginal_table": marginal_table,
        "state_table": state_table,
        "X_obs": X_obs,
        "X_corr_all": X_corr_all,
        "X_changed": X_changed,
        "X_valid": X_valid,
        "valid_rows_mask_raw": is_valid_raw,
        "valid_rows_mask_after_change": is_valid_after_change,
        "support_states": support_states,
        "support_masks": support_masks,
        "obs_counts": obs_counts,
        "D_boot": D_boot,
    }


def print_report_correct_only_invalid(result, top_k=20, digits=4):
    s = result["summary"]

    print("=== TEST multG(1,y): ZMIENIAMY TYLKO OBSERWACJE SPOZA NOŚNIKA ===")
    print(f"csv_path                           = {s['csv_path']}")
    print(f"graph                              = {s['graph']}")
    print(f"edges_python                       = {s['edges_python']}")
    print(f"n_total                            = {s['n_total']}")
    print(f"n_valid_raw                        = {s['n_valid_raw']}")
    print(f"n_invalid_raw                      = {s['n_invalid_raw']}")
    print(f"n_valid_after_change_and_filter    = {s['n_valid_after_change_and_filter']}")
    print(f"n_invalid_after_change_and_filter  = {s['n_invalid_after_change_and_filter']}")
    print(f"n_corrected_invalid_to_valid       = {s['n_corrected_invalid_to_valid']}")
    print(f"n_still_invalid_after_change       = {s['n_still_invalid_after_change']}")
    print(f"frac_valid_after_change_and_filter = {s['frac_valid_after_change_and_filter']:.4f}")
    print(f"support_size                       = {s['support_size']}")
    print(f"deviance_obs                       = {s['deviance_obs']:.4f}")
    print(f"bootstrap_p_value                  = {s['bootstrap_p_value']:.4f}")
    print(f"reject_multG_at_05                 = {s['reject_multG_at_05']}")
    print(f"optimizer_success                  = {s['optimizer_success']}")
    print(f"optimizer_message                  = {s['optimizer_message']}")
    print()

    print("=== MARGINALS ===")
    print(result["marginal_table"].round(digits).to_string(index=False))
    print()

    print(f"=== TOP {top_k} STANÓW Z NOŚNIKA PO ZMIANIE TYLKO INVALID ===")
    print(result["state_table"].head(top_k).round(digits).to_string(index=False))


# ============================================================
# URUCHOMIENIE
# ============================================================

result = test_correct_only_invalid_then_filter_multG1(
    csv_path=CSV_PATH,
    alpha=ALPHA,
    beta=BETA,
    bootstrap_B=BOOTSTRAP_B,
    refit_in_bootstrap=REFIT_IN_BOOTSTRAP,
    seed=SEED,
)

print_report_correct_only_invalid(result, top_k=20, digits=4)

=== TEST multG(1,y): ZMIENIAMY TYLKO OBSERWACJE SPOZA NOŚNIKA ===
csv_path                           = experiment_data_fig2d.csv
graph                              = path4: edges (1,2), (2,3), (3,4)
edges_python                       = [(0, 1), (1, 2), (2, 3)]
n_total                            = 678
n_valid_raw                        = 191
n_invalid_raw                      = 487
n_valid_after_change_and_filter    = 191
n_invalid_after_change_and_filter  = 487
n_corrected_invalid_to_valid       = 0
n_still_invalid_after_change       = 487
frac_valid_after_change_and_filter = 0.2817
support_size                       = 8
deviance_obs                       = 0.5081
bootstrap_p_value                  = 0.9102
reject_multG_at_05                 = False
optimizer_success                  = True
optimizer_message                  = CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*EPSMCH

=== MARGINALS ===
vertex  obs_mean_raw  changed_mean_valid_only  fit_mean  diff   y_hat  theta_hat  alpha  

In [5]:
# ============================================================
# ZMIANA STANÓW NA PODSTAWIE WYESTYMOWANEJ MACIERZY PRZEJŚCIA
# i test multG na zmienionych danych
# ============================================================

def posterior_latent_given_observed(fit_em):
    """
    post[k, m] = P(Z = latent_k | X = observed_m)
    k = latent state index
    m = observed state index
    """
    pi_hat = np.asarray(fit_em["latent_probs_hat"], dtype=float)      # (K,)
    T_hat = np.asarray(fit_em["transition_hat"], dtype=float)         # (K, M)
    obs_probs_hat = np.asarray(fit_em["obs_probs_hat"], dtype=float)  # (M,)

    post = (pi_hat[:, None] * T_hat) / np.clip(obs_probs_hat[None, :], 1e-300, None)
    post = post / post.sum(axis=0, keepdims=True)
    return post


def change_states_using_transition_matrix(X_obs, fit_em, latent_masks, seed=123):
    """
    Dla każdej obserwacji x:
      - patrzymy na posterior P(Z=z | X=x)
      - losujemy nowy latentny stan z tego posterioru

    To jest właściwa "zmiana stanów na podstawie wyestymowanej macierzy przejścia".
    """
    rng = np.random.default_rng(seed)

    obs_masks = matrix_to_masks(X_obs)
    post = posterior_latent_given_observed(fit_em)   # shape (K, M)

    sampled_latent_idx = np.empty(len(obs_masks), dtype=int)
    for i, m in enumerate(obs_masks):
        sampled_latent_idx[i] = rng.choice(post.shape[0], p=post[:, m])

    changed_latent_masks = latent_masks[sampled_latent_idx]
    X_changed = masks_to_matrix(changed_latent_masks, P)

    return X_changed, changed_latent_masks, sampled_latent_idx, post


def fit_multG1_from_state_counts(
    state_counts,
    support_states,
    theta_init=None,
    theta_bounds=(-20.0, 20.0),
):
    counts = np.asarray(state_counts, dtype=float)
    S = np.asarray(support_states, dtype=float)

    n = counts.sum()
    if n <= 0:
        raise ValueError("Brak obserwacji do dopasowania.")

    sufficient_stat = counts @ S
    p = S.shape[1]

    if theta_init is None:
        theta_init = np.zeros(p, dtype=float)
    else:
        theta_init = np.asarray(theta_init, dtype=float)

    def objective(theta):
        eta = S @ theta
        logZ = logsumexp(eta)
        probs = np.exp(eta - logZ)

        nll = n * logZ - sufficient_stat @ theta
        grad = n * (probs @ S) - sufficient_stat
        return nll, grad

    lo, hi = theta_bounds
    bounds = [(lo, hi)] * p

    opt = minimize(
        fun=lambda th: objective(th)[0],
        x0=theta_init,
        jac=lambda th: objective(th)[1],
        method="L-BFGS-B",
        bounds=bounds,
    )

    theta_hat = opt.x
    eta_hat = S @ theta_hat
    logZ_hat = logsumexp(eta_hat)
    probs_hat = np.exp(eta_hat - logZ_hat)

    return {
        "theta_hat": theta_hat,
        "y_hat": np.exp(theta_hat),
        "probs_hat": probs_hat,
        "expected_state_counts": n * probs_hat,
        "fitted_marginals": probs_hat @ S,
        "loglik_hat": float(sufficient_stat @ theta_hat - n * logZ_hat),
        "optimizer_success": bool(opt.success),
        "optimizer_message": str(opt.message),
    }


def deviance_statistic(counts, probs):
    counts = np.asarray(counts, dtype=float)
    probs = np.asarray(probs, dtype=float)

    n = counts.sum()
    expected = np.clip(n * probs, 1e-300, None)

    mask = counts > 0
    return float(2.0 * np.sum(counts[mask] * np.log(counts[mask] / expected[mask])))


def test_multG_after_state_change_from_transition_matrix(
    csv_path,
    bootstrap_B=300,
    refit_in_bootstrap=True,
    seed=123,
):
    # --------------------------------------------------------
    # 1. EM na surowych danych
    # --------------------------------------------------------
    em_result = run_em_star4_full_state_transition(csv_path)
    fit_em = em_result["fit"]

    # --------------------------------------------------------
    # 2. Zmieniamy stany na podstawie wyestymowanej macierzy przejścia
    # --------------------------------------------------------
    X_obs = read_experiment_data(csv_path)

    latent_masks = enumerate_independent_set_masks(P, EDGES)
    latent_states = masks_to_matrix(latent_masks, P)
    latent_mask_to_index = {int(m): i for i, m in enumerate(latent_masks.tolist())}

    X_changed, changed_latent_masks, sampled_latent_idx, post = change_states_using_transition_matrix(
        X_obs=X_obs,
        fit_em=fit_em,
        latent_masks=latent_masks,
        seed=seed,
    )

    # wszystkie X_changed są już w nośniku
    changed_counts = np.zeros(len(latent_masks), dtype=int)
    for m in changed_latent_masks:
        changed_counts[latent_mask_to_index[int(m)]] += 1

    # --------------------------------------------------------
    # 3. Dopasowanie multG do zmienionych danych
    # --------------------------------------------------------
    fit_mult = fit_multG1_from_state_counts(
        state_counts=changed_counts,
        support_states=latent_states,
    )

    D_obs = deviance_statistic(changed_counts, fit_mult["probs_hat"])

    # --------------------------------------------------------
    # 4. Bootstrap
    # --------------------------------------------------------
    rng = np.random.default_rng(seed)
    D_boot = np.empty(bootstrap_B, dtype=float)

    for b in range(bootstrap_B):
        boot_counts = rng.multinomial(len(X_changed), fit_mult["probs_hat"])

        if refit_in_bootstrap:
            fit_b = fit_multG1_from_state_counts(
                state_counts=boot_counts,
                support_states=latent_states,
                theta_init=fit_mult["theta_hat"],
            )
            D_boot[b] = deviance_statistic(boot_counts, fit_b["probs_hat"])
        else:
            D_boot[b] = deviance_statistic(boot_counts, fit_mult["probs_hat"])

    p_value = (1.0 + np.sum(D_boot >= D_obs)) / (bootstrap_B + 1.0)

    # --------------------------------------------------------
    # 5. Tabele
    # --------------------------------------------------------
    changed_state_table = pd.DataFrame({
        "latent_mask": latent_masks,
        "latent_state": [mask_to_bitstring(int(m), P) for m in latent_masks],
        "obs_after_change": changed_counts,
        "exp_under_multG": fit_mult["expected_state_counts"],
        "prob_hat_multG": fit_mult["probs_hat"],
        "em_latent_prob_hat": fit_em["latent_probs_hat"],
    }).sort_values(["obs_after_change", "exp_under_multG"], ascending=[False, False]).reset_index(drop=True)

    marginal_table = pd.DataFrame({
        "vertex": ["1", "2", "3", "4"],
        "mean_after_change": X_changed.mean(axis=0),
        "fit_mean_multG": fit_mult["fitted_marginals"],
        "diff": X_changed.mean(axis=0) - fit_mult["fitted_marginals"],
        "y_hat_multG": fit_mult["y_hat"],
        "theta_hat_multG": fit_mult["theta_hat"],
    })

    obs_masks, obs_states = all_binary_patterns(P)
    mapping_summary = pd.DataFrame({
        "obs_mask": obs_masks,
        "obs_state": [mask_to_bitstring(int(m), P) for m in obs_masks],
        "max_post_prob": np.max(post, axis=0),
        "map_latent_state": [
            mask_to_bitstring(int(latent_masks[np.argmax(post[:, m])]), P)
            for m in range(len(obs_masks))
        ],
    }).sort_values(["max_post_prob", "obs_mask"], ascending=[False, True]).reset_index(drop=True)

    return {
        "summary": {
            "csv_path": csv_path,
            "graph": "star4: edges (1,2), (1,3), (1,4)",
            "edges_python": EDGES,
            "n_total": int(len(X_obs)),
            "support_size": int(len(latent_masks)),
            "deviance_obs_after_state_change": float(D_obs),
            "bootstrap_p_value_after_state_change": float(p_value),
            "reject_multG_at_05_after_state_change": bool(p_value < 0.05),
            "optimizer_success_multG": bool(fit_mult["optimizer_success"]),
            "optimizer_message_multG": fit_mult["optimizer_message"],
            "em_loglik": float(em_result["summary"]["loglik"]),
            "em_iterations": int(em_result["summary"]["em_iterations"]),
        },
        "em_result": em_result,
        "fit_mult_after_change": fit_mult,
        "X_obs": X_obs,
        "X_changed": X_changed,
        "changed_state_table": changed_state_table,
        "marginal_table": marginal_table,
        "mapping_summary": mapping_summary,
        "D_boot": D_boot,
    }


def print_report_mult_after_state_change(result, top_k=20, digits=4):
    s = result["summary"]

    print("=== TEST multG(1,y) PO ZMIANIE STANÓW NA PODSTAWIE WYESTYMOWANEJ MACIERZY PRZEJŚCIA ===")
    print(f"csv_path                               = {s['csv_path']}")
    print(f"graph                                  = {s['graph']}")
    print(f"edges_python                           = {s['edges_python']}")
    print(f"n_total                                = {s['n_total']}")
    print(f"support_size                           = {s['support_size']}")
    print(f"em_loglik                              = {s['em_loglik']:.4f}")
    print(f"em_iterations                          = {s['em_iterations']}")
    print(f"deviance_obs_after_state_change        = {s['deviance_obs_after_state_change']:.4f}")
    print(f"bootstrap_p_value_after_state_change   = {s['bootstrap_p_value_after_state_change']:.4f}")
    print(f"reject_multG_at_05_after_state_change  = {s['reject_multG_at_05_after_state_change']}")
    print(f"optimizer_success_multG                = {s['optimizer_success_multG']}")
    print(f"optimizer_message_multG                = {s['optimizer_message_multG']}")
    print()

    print("=== MARGINALS PO ZMIANIE STANÓW ===")
    print(result["marginal_table"].round(digits).to_string(index=False))
    print()

    print(f"=== TOP {top_k} STANÓW PO ZMIANIE ===")
    print(result["changed_state_table"].head(top_k).round(digits).to_string(index=False))
    print()

    print(f"=== PODSUMOWANIE posterior P(Z|X) ===")
    print(result["mapping_summary"].head(top_k).round(digits).to_string(index=False))


# ============================================================
# URUCHOMIENIE
# ============================================================

result_after_change = test_multG_after_state_change_from_transition_matrix(
    csv_path=CSV_PATH,
    bootstrap_B=BOOTSTRAP_B,
    refit_in_bootstrap=True,
    seed=SEED,
)

print_report_mult_after_state_change(result_after_change, top_k=25, digits=4)

=== TEST multG(1,y) PO ZMIANIE STANÓW NA PODSTAWIE WYESTYMOWANEJ MACIERZY PRZEJŚCIA ===
csv_path                               = experiment_data_fig2d.csv
graph                                  = star4: edges (1,2), (1,3), (1,4)
edges_python                           = [(0, 1), (0, 2), (0, 3)]
n_total                                = 678
support_size                           = 9
em_loglik                              = -1612.1686
em_iterations                          = 500
deviance_obs_after_state_change        = 3.5533
bootstrap_p_value_after_state_change   = 0.4571
reject_multG_at_05_after_state_change  = False
optimizer_success_multG                = True
optimizer_message_multG                = CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*EPSMCH

=== MARGINALS PO ZMIANIE STANÓW ===
vertex  mean_after_change  fit_mean_multG  diff  y_hat_multG  theta_hat_multG
     1             0.1062          0.1062   0.0       0.9570          -0.0439
     2             0.4395          0.4395   

In [17]:
import numpy as np
import pandas as pd
from scipy.optimize import minimize
from scipy.special import logsumexp


# ============================================================
# USTAWIENIA
# ============================================================

CSV_PATH = "experiment_data_fig2d.csv"

# ------------------------------------------------------------
# TU USTAWIASZ GRAF
# ------------------------------------------------------------
# Przykład: ścieżka 1-2-3-4
GRAPH_NAME = "path4: edges (1,2), (2,3), (3,4)"
EDGES = [
    (0, 1),
    (1, 2),
    (2, 3),
]

P = 4

# ------------------------------------------------------------
# EM
# ------------------------------------------------------------
EM_MAX_ITER = 300
EM_TOL = 1e-8
EM_EPS = 1e-6

# ograniczenia na wspólne parametry błędu
BETA_MIN = 0.03
ALPHA_MIN = 1e-6
ALPHA_MAX = 0.5

# ------------------------------------------------------------
# bootstrap
# ------------------------------------------------------------
BOOTSTRAP_B = 300
REFIT_EM_IN_BOOTSTRAP = True
REFIT_MULTG_IN_BOOTSTRAP = True
SEED = 123

# ------------------------------------------------------------
# inicjalizacja błędów odczytu
# ------------------------------------------------------------
ALPHA_INIT = 0.10
BETA_INIT = 0.05

# ------------------------------------------------------------
# checkpointy
# ------------------------------------------------------------
VERBOSE = True


def checkpoint(msg, verbose=VERBOSE):
    if verbose:
        print(msg, flush=True)


# ============================================================
# WCZYTANIE DANYCH
# ============================================================

def read_experiment_data(csv_path):
    df = pd.read_csv(csv_path, header=None)
    df = df.dropna(axis=0, how="all").dropna(axis=1, how="all")

    if df.shape[0] < P:
        raise ValueError(
            f"Oczekiwałem co najmniej {P} wierszy w pliku {csv_path}, a jest {df.shape[0]}."
        )

    X_raw = df.iloc[:P, :].to_numpy()

    uniq = np.unique(X_raw)
    if not np.all(np.isin(uniq, [0, 1])):
        raise ValueError(f"Dane muszą być binarne 0/1. Unikalne wartości: {uniq}")

    return X_raw.T.astype(int)


# ============================================================
# NARZĘDZIA STANÓW
# ============================================================

def matrix_to_masks(X):
    p = X.shape[1]
    powers = (1 << np.arange(p, dtype=np.int64))
    return (X.astype(np.int64) * powers).sum(axis=1)


def masks_to_matrix(masks, p):
    masks = np.asarray(masks, dtype=np.int64)
    return ((masks[:, None] >> np.arange(p, dtype=np.int64)) & 1).astype(int)


def mask_to_bitstring(mask, p):
    return "".join(str((mask >> j) & 1) for j in range(p))


def all_binary_patterns(p):
    masks = np.arange(1 << p, dtype=np.int64)
    states = masks_to_matrix(masks, p)
    return masks, states


def counts_over_all_patterns(X):
    all_masks, all_states = all_binary_patterns(X.shape[1])
    obs_masks = matrix_to_masks(X)
    counts = np.zeros(len(all_masks), dtype=int)
    for m in obs_masks:
        counts[int(m)] += 1
    return all_masks, all_states, counts


def valid_rows_mask(X, edges):
    ok = np.ones(X.shape[0], dtype=bool)
    for u, v in edges:
        ok &= ~((X[:, u] == 1) & (X[:, v] == 1))
    return ok


def enumerate_independent_set_masks(p, edges):
    masks = []
    for mask in range(1 << p):
        good = True
        for u, v in edges:
            if ((mask >> u) & 1) and ((mask >> v) & 1):
                good = False
                break
        if good:
            masks.append(mask)
    return np.array(masks, dtype=np.int64)


# ============================================================
# multG(1,y) NA NOŚNIKU
# ============================================================

def fit_multG1_from_state_counts(
    state_counts,
    support_states,
    theta_init=None,
    theta_bounds=(-20.0, 20.0),
):
    counts = np.asarray(state_counts, dtype=float)
    S = np.asarray(support_states, dtype=float)

    n = counts.sum()
    if n <= 0:
        raise ValueError("Brak obserwacji do dopasowania.")

    sufficient_stat = counts @ S
    p = S.shape[1]

    if theta_init is None:
        theta_init = np.zeros(p, dtype=float)
    else:
        theta_init = np.asarray(theta_init, dtype=float)

    def objective(theta):
        eta = S @ theta
        logZ = logsumexp(eta)
        probs = np.exp(eta - logZ)

        nll = n * logZ - sufficient_stat @ theta
        grad = n * (probs @ S) - sufficient_stat
        return nll, grad

    lo, hi = theta_bounds
    bounds = [(lo, hi)] * p

    opt = minimize(
        fun=lambda th: objective(th)[0],
        x0=theta_init,
        jac=lambda th: objective(th)[1],
        method="L-BFGS-B",
        bounds=bounds,
    )

    theta_hat = opt.x
    eta_hat = S @ theta_hat
    logZ_hat = logsumexp(eta_hat)
    probs_hat = np.exp(eta_hat - logZ_hat)

    return {
        "theta_hat": theta_hat,
        "y_hat": np.exp(theta_hat),
        "probs_hat": probs_hat,
        "expected_state_counts": n * probs_hat,
        "fitted_marginals": probs_hat @ S,
        "loglik_hat": float(sufficient_stat @ theta_hat - n * logZ_hat),
        "optimizer_success": bool(opt.success),
        "optimizer_message": str(opt.message),
    }


# ============================================================
# MODEL SZUMU BITOWEGO
# ============================================================

def emission_matrix_from_alpha_beta(latent_states, observed_states, alpha, beta):
    """
    T[k,m] = P(X = observed_m | Z = latent_k)
    przy wspólnych alpha i beta dla wszystkich wierzchołków.
    """
    latent_states = np.asarray(latent_states, dtype=int)
    observed_states = np.asarray(observed_states, dtype=int)

    K, p = latent_states.shape
    M = observed_states.shape[0]

    T = np.ones((K, M), dtype=float)

    for j in range(p):
        z = latent_states[:, j][:, None]
        x = observed_states[:, j][None, :]

        part0 = np.where(x == 1, alpha, 1 - alpha)
        part1 = np.where(x == 0, beta, 1 - beta)

        T *= np.where(z == 0, part0, part1)

    T = np.clip(T, 1e-300, None)
    return T


# ============================================================
# EM
# ============================================================

def em_fit_multG_bitflip(
    observed_counts,
    observed_states,
    latent_states,
    alpha_init,
    beta_init,
    max_iter=300,
    tol=1e-8,
    eps=1e-6,
    alpha_min=1e-6,
    alpha_max=0.5,
    beta_min=0.03,
    verbose=VERBOSE,
):
    obs_counts = np.asarray(observed_counts, dtype=float)
    observed_states = np.asarray(observed_states, dtype=int)
    latent_states = np.asarray(latent_states, dtype=int)

    K = latent_states.shape[0]
    M = observed_states.shape[0]
    p = latent_states.shape[1]

    alpha = float(alpha_init)
    beta = float(beta_init)
    theta = np.zeros(p, dtype=float)

    loglik_trace = []

    checkpoint("=== START EM ===", verbose)
    checkpoint(f"K={K}, M={M}, p={p}", verbose)
    checkpoint(f"alpha_init={alpha:.4f}", verbose)
    checkpoint(f"beta_init ={beta:.4f}", verbose)
    checkpoint(f"alpha_max ={alpha_max}", verbose)
    checkpoint(f"beta_min  ={beta_min}", verbose)

    for it in range(max_iter):
        fit_lat = fit_multG1_from_state_counts(
            state_counts=np.ones(K),
            support_states=latent_states,
            theta_init=theta,
        )
        theta = fit_lat["theta_hat"]
        pi = fit_lat["probs_hat"]

        T = emission_matrix_from_alpha_beta(
            latent_states=latent_states,
            observed_states=observed_states,
            alpha=alpha,
            beta=beta,
        )

        obs_probs = np.clip(pi @ T, 1e-300, None)
        loglik = float(np.sum(obs_counts * np.log(obs_probs)))
        loglik_trace.append(loglik)

        R = (pi[:, None] * T) / obs_probs[None, :]
        N_km = R * obs_counts[None, :]
        N_k = N_km.sum(axis=1)

        fit_lat = fit_multG1_from_state_counts(
            state_counts=N_k,
            support_states=latent_states,
            theta_init=theta,
        )
        theta_new = fit_lat["theta_hat"]
        pi_new = fit_lat["probs_hat"]

        Z = latent_states[:, None, :]      # (K, 1, p)
        X = observed_states[None, :, :]    # (1, M, p)

        denom0 = np.sum(N_km[:, :, None] * (1 - Z))
        num01 = np.sum(N_km[:, :, None] * (1 - Z) * X)

        denom1 = np.sum(N_km[:, :, None] * Z)
        num10 = np.sum(N_km[:, :, None] * Z * (1 - X))

        alpha_new = num01 / denom0 if denom0 > 0 else alpha_min
        beta_new = num10 / denom1 if denom1 > 0 else beta_min

        alpha_new = float(np.clip(alpha_new, alpha_min, alpha_max))
        beta_new = float(np.clip(beta_new, beta_min, 1 - eps))

        delta = max(
            np.max(np.abs(theta_new - theta)),
            abs(alpha_new - alpha),
            abs(beta_new - beta),
        )

        if it == 0 or (it + 1) % 10 == 0:
            checkpoint(
                f"[EM] iter={it+1:03d}  loglik={loglik:.6f}  "
                f"delta={delta:.3e}  alpha={alpha_new:.4f}  beta={beta_new:.4f}",
                verbose,
            )

        theta = theta_new
        alpha = alpha_new
        beta = beta_new

        if it > 0 and abs(loglik_trace[-1] - loglik_trace[-2]) < tol and delta < tol:
            checkpoint(f"[EM] stop at iter={it+1}, loglik={loglik:.6f}", verbose)
            break

    T = emission_matrix_from_alpha_beta(
        latent_states=latent_states,
        observed_states=observed_states,
        alpha=alpha,
        beta=beta,
    )
    obs_probs = np.clip(pi_new @ T, 1e-300, None)

    M_common = np.array([
        [1 - alpha, alpha],
        [beta, 1 - beta],
    ])

    checkpoint("=== END EM ===", verbose)

    return {
        "theta_hat": theta,
        "y_hat": np.exp(theta),
        "latent_probs_hat": pi_new,
        "alpha_hat": alpha,
        "beta_hat": beta,
        "M_hat": M_common,
        "transition_hat_statewise": T,
        "obs_probs_hat": obs_probs,
        "expected_obs_counts": observed_counts.sum() * obs_probs,
        "loglik": float(np.sum(observed_counts * np.log(obs_probs))),
        "n_iter": len(loglik_trace),
        "loglik_trace": loglik_trace,
    }


# ============================================================
# POSTERIOR I KOREKTA PO EM
# ============================================================

def posterior_latent_given_observed(fit_em):
    pi_hat = np.asarray(fit_em["latent_probs_hat"], dtype=float)
    T_hat = np.asarray(fit_em["transition_hat_statewise"], dtype=float)
    obs_probs_hat = np.asarray(fit_em["obs_probs_hat"], dtype=float)

    post = (pi_hat[:, None] * T_hat) / np.clip(obs_probs_hat[None, :], 1e-300, None)
    post = post / post.sum(axis=0, keepdims=True)
    return post


def latent_bit_priors_from_fit(fit_em, latent_states):
    """
    prior_true_j = P(Z_j = 1) pod wyestymowanym latentnym multG
    """
    return fit_em["latent_probs_hat"] @ latent_states


def correct_bits_independently_from_em_params(X_obs, alpha, beta, prior_true):
    """
    Czysta korekta bit-po-bicie:
    dla każdego bitu osobno używamy tylko:
      - X_j
      - alpha
      - beta
      - prior_true_j = P(Z_j=1)

    Nie używamy posterioru po całych stanach do ustawiania bitów.
    """
    X_obs = np.asarray(X_obs, dtype=int)
    n, p = X_obs.shape
    X_corr = np.zeros_like(X_obs)

    for j in range(p):
        pi = prior_true[j]

        # P(Z_j=1 | X_j=1)
        post1 = ((1 - beta) * pi) / (((1 - beta) * pi) + alpha * (1 - pi))

        # P(Z_j=1 | X_j=0)
        post0 = (beta * pi) / ((beta * pi) + (1 - alpha) * (1 - pi))

        X_corr[X_obs[:, j] == 1, j] = int(post1 > 0.5)
        X_corr[X_obs[:, j] == 0, j] = int(post0 > 0.5)

    return X_corr


# ============================================================
# STATYSTYKA TESTOWA
# ============================================================

def deviance_statistic(counts, probs):
    counts = np.asarray(counts, dtype=float)
    probs = np.asarray(probs, dtype=float)

    n = counts.sum()
    expected = np.clip(n * probs, 1e-300, None)

    mask = counts > 0
    return float(2.0 * np.sum(counts[mask] * np.log(counts[mask] / expected[mask])))


# ============================================================
# SYMULACJA
# ============================================================

def simulate_from_em_model(n, latent_states, latent_probs, alpha, beta, rng):
    K, p = latent_states.shape

    idx = rng.choice(K, size=n, p=latent_probs)
    Z = latent_states[idx]
    X = np.zeros_like(Z)

    for j in range(p):
        u = rng.random(n)
        z = Z[:, j]

        mask0 = (z == 0)
        X[mask0, j] = (u[mask0] < alpha).astype(int)

        mask1 = (z == 1)
        X[mask1, j] = (u[mask1] >= beta).astype(int)

    return Z, X


# ============================================================
# TEST 1: PO KOREKCIE BIT-PO-BICIE I OBCIĘCIU DO NOŚNIKA
# ============================================================

def test_after_bitwise_correction_and_cut(
    X_obs,
    fit_em,
    latent_masks,
    latent_states,
    bootstrap_B=300,
    refit_multg_in_bootstrap=True,
    seed=123,
    verbose=VERBOSE,
):
    checkpoint("=== TEST CUT: czysta korekta bit-po-bicie i obcięcie do nośnika ===", verbose)

    alpha = fit_em["alpha_hat"]
    beta = fit_em["beta_hat"]
    prior_true = latent_bit_priors_from_fit(fit_em, latent_states)

    X_corr = correct_bits_independently_from_em_params(
        X_obs=X_obs,
        alpha=alpha,
        beta=beta,
        prior_true=prior_true,
    )

    is_valid = valid_rows_mask(X_corr, EDGES)
    X_valid = X_corr[is_valid]

    checkpoint(
        f"[CUT] n_total={len(X_obs)}, n_valid={len(X_valid)}, n_invalid={len(X_obs)-len(X_valid)}",
        verbose,
    )

    if len(X_valid) == 0:
        raise ValueError("Po korekcie i obcięciu do nośnika nie zostały żadne dane.")

    latent_mask_to_index = {int(m): i for i, m in enumerate(latent_masks.tolist())}
    valid_masks = matrix_to_masks(X_valid)

    counts_cut = np.zeros(len(latent_masks), dtype=int)
    for m in valid_masks:
        counts_cut[latent_mask_to_index[int(m)]] += 1

    fit_mult = fit_multG1_from_state_counts(
        state_counts=counts_cut,
        support_states=latent_states,
    )

    D_obs = deviance_statistic(counts_cut, fit_mult["probs_hat"])
    checkpoint(f"[CUT] D_obs={D_obs:.6f}", verbose)

    rng = np.random.default_rng(seed)
    D_boot = np.empty(bootstrap_B, dtype=float)

    for b in range(bootstrap_B):
        boot_counts = rng.multinomial(len(X_valid), fit_mult["probs_hat"])

        if refit_multg_in_bootstrap:
            fit_b = fit_multG1_from_state_counts(
                state_counts=boot_counts,
                support_states=latent_states,
                theta_init=fit_mult["theta_hat"],
            )
            D_boot[b] = deviance_statistic(boot_counts, fit_b["probs_hat"])
        else:
            D_boot[b] = deviance_statistic(boot_counts, fit_mult["probs_hat"])

        if (b + 1) % 50 == 0 or b == 0 or b + 1 == bootstrap_B:
            checkpoint(f"[CUT bootstrap] {b+1}/{bootstrap_B}", verbose)

    p_value = (1.0 + np.sum(D_boot >= D_obs)) / (bootstrap_B + 1.0)
    checkpoint(f"[CUT] p_value={p_value:.6f}", verbose)

    state_table = pd.DataFrame({
        "latent_mask": latent_masks,
        "latent_state": [mask_to_bitstring(int(m), P) for m in latent_masks],
        "obs_after_correction_cut": counts_cut,
        "exp_under_multG": fit_mult["expected_state_counts"],
        "prob_hat_multG": fit_mult["probs_hat"],
    }).sort_values(["obs_after_correction_cut", "exp_under_multG"], ascending=[False, False]).reset_index(drop=True)

    return {
        "summary": {
            "n_total": int(len(X_obs)),
            "n_valid_after_cut": int(len(X_valid)),
            "n_invalid_after_cut": int(len(X_obs) - len(X_valid)),
            "frac_valid_after_cut": float(len(X_valid) / len(X_obs)),
            "deviance_obs_cut": float(D_obs),
            "bootstrap_p_value_cut": float(p_value),
            "reject_multG_at_05_cut": bool(p_value < 0.05),
            "optimizer_success_cut": bool(fit_mult["optimizer_success"]),
            "optimizer_message_cut": fit_mult["optimizer_message"],
        },
        "X_corr": X_corr,
        "X_valid": X_valid,
        "valid_rows_mask_after_cut": is_valid,
        "fit_mult_cut": fit_mult,
        "state_table_cut": state_table,
        "D_boot_cut": D_boot,
    }


# ============================================================
# TEST 2: BEZ OBCINANIA NOŚNIKA
# ============================================================

def test_without_cutting_support(
    X_obs,
    observed_counts,
    observed_states,
    latent_states,
    fit_em,
    bootstrap_B=300,
    refit_em_in_bootstrap=True,
    seed=123,
    verbose=VERBOSE,
):
    checkpoint("=== TEST NO-CUT: pełny model noisy multG ===", verbose)

    D_obs = deviance_statistic(observed_counts, fit_em["obs_probs_hat"])
    checkpoint(f"[NO-CUT] D_obs={D_obs:.6f}", verbose)

    rng = np.random.default_rng(seed)
    D_boot = np.empty(bootstrap_B, dtype=float)

    for b in range(bootstrap_B):
        _, Xb = simulate_from_em_model(
            n=len(X_obs),
            latent_states=latent_states,
            latent_probs=fit_em["latent_probs_hat"],
            alpha=fit_em["alpha_hat"],
            beta=fit_em["beta_hat"],
            rng=rng,
        )

        _, obs_states_b, obs_counts_b = counts_over_all_patterns(Xb)

        if refit_em_in_bootstrap:
            fit_b = em_fit_multG_bitflip(
                observed_counts=obs_counts_b,
                observed_states=obs_states_b,
                latent_states=latent_states,
                alpha_init=fit_em["alpha_hat"],
                beta_init=fit_em["beta_hat"],
                max_iter=EM_MAX_ITER,
                tol=EM_TOL,
                eps=EM_EPS,
                alpha_min=ALPHA_MIN,
                alpha_max=ALPHA_MAX,
                beta_min=BETA_MIN,
                verbose=False,
            )
            D_boot[b] = deviance_statistic(obs_counts_b, fit_b["obs_probs_hat"])
        else:
            D_boot[b] = deviance_statistic(obs_counts_b, fit_em["obs_probs_hat"])

        if (b + 1) % 25 == 0 or b == 0 or b + 1 == bootstrap_B:
            checkpoint(f"[NO-CUT bootstrap] {b+1}/{bootstrap_B}", verbose)

    p_value = (1.0 + np.sum(D_boot >= D_obs)) / (bootstrap_B + 1.0)
    checkpoint(f"[NO-CUT] p_value={p_value:.6f}", verbose)

    obs_table = pd.DataFrame({
        "obs_state": [mask_to_bitstring(i, P) for i in range(2 ** P)],
        "obs_count": observed_counts,
        "exp_under_noisy_model": fit_em["expected_obs_counts"],
        "obs_prob_hat": fit_em["obs_probs_hat"],
    }).sort_values(["obs_count", "exp_under_noisy_model"], ascending=[False, False]).reset_index(drop=True)

    return {
        "summary": {
            "deviance_obs_no_cut": float(D_obs),
            "bootstrap_p_value_no_cut": float(p_value),
            "reject_model_at_05_no_cut": bool(p_value < 0.05),
        },
        "obs_table_no_cut": obs_table,
        "D_boot_no_cut": D_boot,
    }


# ============================================================
# GŁÓWNA FUNKCJA
# ============================================================

def run_full_pipeline_em_bitflip_and_two_tests(
    csv_path,
    alpha_init,
    beta_init,
    bootstrap_B=300,
    refit_em_in_bootstrap=True,
    refit_multg_in_bootstrap=True,
    seed=123,
    verbose=VERBOSE,
):
    checkpoint("=== START PIPELINE ===", verbose)

    X_obs = read_experiment_data(csv_path)
    checkpoint(f"[PIPELINE] loaded X_obs with n={len(X_obs)}", verbose)

    obs_masks, observed_states, observed_counts = counts_over_all_patterns(X_obs)

    latent_masks = enumerate_independent_set_masks(P, EDGES)
    latent_states = masks_to_matrix(latent_masks, P)
    checkpoint(f"[PIPELINE] latent support size={len(latent_masks)}", verbose)

    fit_em = em_fit_multG_bitflip(
        observed_counts=observed_counts,
        observed_states=observed_states,
        latent_states=latent_states,
        alpha_init=alpha_init,
        beta_init=beta_init,
        max_iter=EM_MAX_ITER,
        tol=EM_TOL,
        eps=EM_EPS,
        alpha_min=ALPHA_MIN,
        alpha_max=ALPHA_MAX,
        beta_min=BETA_MIN,
        verbose=verbose,
    )

    test_cut = test_after_bitwise_correction_and_cut(
        X_obs=X_obs,
        fit_em=fit_em,
        latent_masks=latent_masks,
        latent_states=latent_states,
        bootstrap_B=bootstrap_B,
        refit_multg_in_bootstrap=refit_multg_in_bootstrap,
        seed=seed,
        verbose=verbose,
    )

    test_no_cut = test_without_cutting_support(
        X_obs=X_obs,
        observed_counts=observed_counts,
        observed_states=observed_states,
        latent_states=latent_states,
        fit_em=fit_em,
        bootstrap_B=bootstrap_B,
        refit_em_in_bootstrap=refit_em_in_bootstrap,
        seed=seed,
        verbose=verbose,
    )

    checkpoint("=== END PIPELINE ===", verbose)

    latent_table = pd.DataFrame({
        "latent_mask": latent_masks,
        "latent_state": [mask_to_bitstring(int(m), P) for m in latent_masks],
        "latent_prob_hat": fit_em["latent_probs_hat"],
    }).sort_values("latent_prob_hat", ascending=False).reset_index(drop=True)

    return {
        "summary": {
            "csv_path": csv_path,
            "graph": GRAPH_NAME,
            "edges_python": EDGES,
            "n_total": int(len(X_obs)),
            "support_size": int(len(latent_masks)),
            "em_loglik": float(fit_em["loglik"]),
            "em_iterations": int(fit_em["n_iter"]),
            "beta_min_used": float(BETA_MIN),
            "alpha_max_used": float(ALPHA_MAX),
        },
        "fit_em": fit_em,
        "latent_table": latent_table,
        "test_cut": test_cut,
        "test_no_cut": test_no_cut,
    }


def print_full_pipeline_report(result, top_k=20, digits=4):
    s = result["summary"]
    sc = result["test_cut"]["summary"]
    sn = result["test_no_cut"]["summary"]

    print("=== EM + DWA TESTY ===")
    print(f"csv_path                = {s['csv_path']}")
    print(f"graph                   = {s['graph']}")
    print(f"edges_python            = {s['edges_python']}")
    print(f"n_total                 = {s['n_total']}")
    print(f"support_size            = {s['support_size']}")
    print(f"em_loglik               = {s['em_loglik']:.4f}")
    print(f"em_iterations           = {s['em_iterations']}")
    print(f"beta_min_used           = {s['beta_min_used']:.4f}")
    print(f"alpha_max_used          = {s['alpha_max_used']:.4f}")
    print()

    print("=== WSPÓLNA BITOWA MACIERZ M ===")
    print(pd.DataFrame(
        result["fit_em"]["M_hat"],
        index=["true0", "true1"],
        columns=["obs0", "obs1"]
    ).round(digits).to_string())
    print()
    print(f"alpha_hat = {result['fit_em']['alpha_hat']:.4f}")
    print(f"beta_hat  = {result['fit_em']['beta_hat']:.4f}")
    print()

    print("=== TEST 1: PO KOREKCIE I OBCIĘCIU DO NOŚNIKA ===")
    print(f"n_valid_after_cut       = {sc['n_valid_after_cut']}")
    print(f"n_invalid_after_cut     = {sc['n_invalid_after_cut']}")
    print(f"frac_valid_after_cut    = {sc['frac_valid_after_cut']:.4f}")
    print(f"deviance_obs_cut        = {sc['deviance_obs_cut']:.4f}")
    print(f"bootstrap_p_value_cut   = {sc['bootstrap_p_value_cut']:.4f}")
    print(f"reject_multG_at_05_cut  = {sc['reject_multG_at_05_cut']}")
    print()

    print("=== TEST 2: BEZ OBCINANIA NOŚNIKA ===")
    print(f"deviance_obs_no_cut      = {sn['deviance_obs_no_cut']:.4f}")
    print(f"bootstrap_p_value_no_cut = {sn['bootstrap_p_value_no_cut']:.4f}")
    print(f"reject_model_at_05_no_cut= {sn['reject_model_at_05_no_cut']}")
    print()

    print(f"=== TOP {top_k} LATENTNYCH STANÓW ===")
    print(result["latent_table"].head(top_k).round(digits).to_string(index=False))
    print()

    print(f"=== TOP {top_k} STANÓW PO KOREKCIE I OBCIĘCIU ===")
    print(result["test_cut"]["state_table_cut"].head(top_k).round(digits).to_string(index=False))
    print()

    print(f"=== TOP {top_k} STANÓW OBSERWOWANYCH (NO-CUT) ===")
    print(result["test_no_cut"]["obs_table_no_cut"].head(top_k).round(digits).to_string(index=False))


# ============================================================
# URUCHOMIENIE
# ============================================================

result = run_full_pipeline_em_bitflip_and_two_tests(
    csv_path=CSV_PATH,
    alpha_init=ALPHA_INIT,
    beta_init=BETA_INIT,
    bootstrap_B=BOOTSTRAP_B,
    refit_em_in_bootstrap=REFIT_EM_IN_BOOTSTRAP,
    refit_multg_in_bootstrap=REFIT_MULTG_IN_BOOTSTRAP,
    seed=SEED,
    verbose=True,
)

print_full_pipeline_report(result, top_k=25, digits=4)

=== START PIPELINE ===
[PIPELINE] loaded X_obs with n=678
[PIPELINE] latent support size=8
=== START EM ===
K=8, M=16, p=4
alpha_init=0.1000
beta_init =0.0500
alpha_max =0.5
beta_min  =0.03
[EM] iter=001  loglik=-2473.360859  delta=1.509e+00  alpha=0.4037  beta=0.0300
[EM] iter=010  loglik=-1793.630680  delta=3.414e-01  alpha=0.4823  beta=0.0300
[EM] iter=020  loglik=-1793.630680  delta=3.414e-01  alpha=0.4823  beta=0.0300
[EM] iter=030  loglik=-1793.630680  delta=3.414e-01  alpha=0.4823  beta=0.0300
[EM] iter=040  loglik=-1793.630680  delta=3.414e-01  alpha=0.4823  beta=0.0300
[EM] iter=050  loglik=-1793.630680  delta=3.414e-01  alpha=0.4823  beta=0.0300
[EM] iter=060  loglik=-1793.630680  delta=3.414e-01  alpha=0.4823  beta=0.0300
[EM] iter=070  loglik=-1793.630680  delta=3.414e-01  alpha=0.4823  beta=0.0300
[EM] iter=080  loglik=-1793.630680  delta=3.414e-01  alpha=0.4823  beta=0.0300
[EM] iter=090  loglik=-1793.630680  delta=3.414e-01  alpha=0.4823  beta=0.0300
[EM] iter=100  logli

In [18]:
import numpy as np
import pandas as pd
from scipy.optimize import minimize
from scipy.special import logsumexp


# ============================================================
# USTAWIENIA
# ============================================================

CSV_PATH = "experiment_data_fig2d.csv"

# ------------------------------------------------------------
# TU USTAWIASZ GRAF
# ------------------------------------------------------------
# Przykład: ścieżka 1-2-3-4
GRAPH_NAME = "path4: edges (1,3), (2,3), (3,4)"
EDGES = [
    (0, 2),
    (1, 2),
    (2, 3),
]

P = 4

# ------------------------------------------------------------
# EM
# ------------------------------------------------------------
EM_MAX_ITER = 300
EM_TOL = 1e-8
EM_EPS = 1e-6

# ograniczenia na wspólne parametry błędu
BETA_MIN = 0.03
ALPHA_MIN = 1e-6
ALPHA_MAX = 0.5

# ------------------------------------------------------------
# bootstrap
# ------------------------------------------------------------
BOOTSTRAP_B = 300
REFIT_EM_IN_BOOTSTRAP = True
REFIT_MULTG_IN_BOOTSTRAP = True
SEED = 123

# ------------------------------------------------------------
# inicjalizacja błędów odczytu
# ------------------------------------------------------------
ALPHA_INIT = 0.10
BETA_INIT = 0.05

# ------------------------------------------------------------
# checkpointy
# ------------------------------------------------------------
VERBOSE = True


def checkpoint(msg, verbose=VERBOSE):
    if verbose:
        print(msg, flush=True)


# ============================================================
# WCZYTANIE DANYCH
# ============================================================

def read_experiment_data(csv_path):
    df = pd.read_csv(csv_path, header=None)
    df = df.dropna(axis=0, how="all").dropna(axis=1, how="all")

    if df.shape[0] < P:
        raise ValueError(
            f"Oczekiwałem co najmniej {P} wierszy w pliku {csv_path}, a jest {df.shape[0]}."
        )

    X_raw = df.iloc[:P, :].to_numpy()

    uniq = np.unique(X_raw)
    if not np.all(np.isin(uniq, [0, 1])):
        raise ValueError(f"Dane muszą być binarne 0/1. Unikalne wartości: {uniq}")

    return X_raw.T.astype(int)


# ============================================================
# NARZĘDZIA STANÓW
# ============================================================

def matrix_to_masks(X):
    p = X.shape[1]
    powers = (1 << np.arange(p, dtype=np.int64))
    return (X.astype(np.int64) * powers).sum(axis=1)


def masks_to_matrix(masks, p):
    masks = np.asarray(masks, dtype=np.int64)
    return ((masks[:, None] >> np.arange(p, dtype=np.int64)) & 1).astype(int)


def mask_to_bitstring(mask, p):
    return "".join(str((mask >> j) & 1) for j in range(p))


def all_binary_patterns(p):
    masks = np.arange(1 << p, dtype=np.int64)
    states = masks_to_matrix(masks, p)
    return masks, states


def counts_over_all_patterns(X):
    all_masks, all_states = all_binary_patterns(X.shape[1])
    obs_masks = matrix_to_masks(X)
    counts = np.zeros(len(all_masks), dtype=int)
    for m in obs_masks:
        counts[int(m)] += 1
    return all_masks, all_states, counts


def valid_rows_mask(X, edges):
    ok = np.ones(X.shape[0], dtype=bool)
    for u, v in edges:
        ok &= ~((X[:, u] == 1) & (X[:, v] == 1))
    return ok


def enumerate_independent_set_masks(p, edges):
    masks = []
    for mask in range(1 << p):
        good = True
        for u, v in edges:
            if ((mask >> u) & 1) and ((mask >> v) & 1):
                good = False
                break
        if good:
            masks.append(mask)
    return np.array(masks, dtype=np.int64)


# ============================================================
# multG(1,y) NA NOŚNIKU
# ============================================================

def fit_multG1_from_state_counts(
    state_counts,
    support_states,
    theta_init=None,
    theta_bounds=(-20.0, 20.0),
):
    counts = np.asarray(state_counts, dtype=float)
    S = np.asarray(support_states, dtype=float)

    n = counts.sum()
    if n <= 0:
        raise ValueError("Brak obserwacji do dopasowania.")

    sufficient_stat = counts @ S
    p = S.shape[1]

    if theta_init is None:
        theta_init = np.zeros(p, dtype=float)
    else:
        theta_init = np.asarray(theta_init, dtype=float)

    def objective(theta):
        eta = S @ theta
        logZ = logsumexp(eta)
        probs = np.exp(eta - logZ)

        nll = n * logZ - sufficient_stat @ theta
        grad = n * (probs @ S) - sufficient_stat
        return nll, grad

    lo, hi = theta_bounds
    bounds = [(lo, hi)] * p

    opt = minimize(
        fun=lambda th: objective(th)[0],
        x0=theta_init,
        jac=lambda th: objective(th)[1],
        method="L-BFGS-B",
        bounds=bounds,
    )

    theta_hat = opt.x
    eta_hat = S @ theta_hat
    logZ_hat = logsumexp(eta_hat)
    probs_hat = np.exp(eta_hat - logZ_hat)

    return {
        "theta_hat": theta_hat,
        "y_hat": np.exp(theta_hat),
        "probs_hat": probs_hat,
        "expected_state_counts": n * probs_hat,
        "fitted_marginals": probs_hat @ S,
        "loglik_hat": float(sufficient_stat @ theta_hat - n * logZ_hat),
        "optimizer_success": bool(opt.success),
        "optimizer_message": str(opt.message),
    }


# ============================================================
# MODEL SZUMU BITOWEGO
# ============================================================

def emission_matrix_from_alpha_beta(latent_states, observed_states, alpha, beta):
    """
    T[k,m] = P(X = observed_m | Z = latent_k)
    przy wspólnych alpha i beta dla wszystkich wierzchołków.
    """
    latent_states = np.asarray(latent_states, dtype=int)
    observed_states = np.asarray(observed_states, dtype=int)

    K, p = latent_states.shape
    M = observed_states.shape[0]

    T = np.ones((K, M), dtype=float)

    for j in range(p):
        z = latent_states[:, j][:, None]
        x = observed_states[:, j][None, :]

        part0 = np.where(x == 1, alpha, 1 - alpha)
        part1 = np.where(x == 0, beta, 1 - beta)

        T *= np.where(z == 0, part0, part1)

    T = np.clip(T, 1e-300, None)
    return T


# ============================================================
# EM
# ============================================================

def em_fit_multG_bitflip(
    observed_counts,
    observed_states,
    latent_states,
    alpha_init,
    beta_init,
    max_iter=300,
    tol=1e-8,
    eps=1e-6,
    alpha_min=1e-6,
    alpha_max=0.5,
    beta_min=0.03,
    verbose=VERBOSE,
):
    obs_counts = np.asarray(observed_counts, dtype=float)
    observed_states = np.asarray(observed_states, dtype=int)
    latent_states = np.asarray(latent_states, dtype=int)

    K = latent_states.shape[0]
    M = observed_states.shape[0]
    p = latent_states.shape[1]

    alpha = float(alpha_init)
    beta = float(beta_init)
    theta = np.zeros(p, dtype=float)

    loglik_trace = []

    checkpoint("=== START EM ===", verbose)
    checkpoint(f"K={K}, M={M}, p={p}", verbose)
    checkpoint(f"alpha_init={alpha:.4f}", verbose)
    checkpoint(f"beta_init ={beta:.4f}", verbose)
    checkpoint(f"alpha_max ={alpha_max}", verbose)
    checkpoint(f"beta_min  ={beta_min}", verbose)

    for it in range(max_iter):
        fit_lat = fit_multG1_from_state_counts(
            state_counts=np.ones(K),
            support_states=latent_states,
            theta_init=theta,
        )
        theta = fit_lat["theta_hat"]
        pi = fit_lat["probs_hat"]

        T = emission_matrix_from_alpha_beta(
            latent_states=latent_states,
            observed_states=observed_states,
            alpha=alpha,
            beta=beta,
        )

        obs_probs = np.clip(pi @ T, 1e-300, None)
        loglik = float(np.sum(obs_counts * np.log(obs_probs)))
        loglik_trace.append(loglik)

        R = (pi[:, None] * T) / obs_probs[None, :]
        N_km = R * obs_counts[None, :]
        N_k = N_km.sum(axis=1)

        fit_lat = fit_multG1_from_state_counts(
            state_counts=N_k,
            support_states=latent_states,
            theta_init=theta,
        )
        theta_new = fit_lat["theta_hat"]
        pi_new = fit_lat["probs_hat"]

        Z = latent_states[:, None, :]      # (K, 1, p)
        X = observed_states[None, :, :]    # (1, M, p)

        denom0 = np.sum(N_km[:, :, None] * (1 - Z))
        num01 = np.sum(N_km[:, :, None] * (1 - Z) * X)

        denom1 = np.sum(N_km[:, :, None] * Z)
        num10 = np.sum(N_km[:, :, None] * Z * (1 - X))

        alpha_new = num01 / denom0 if denom0 > 0 else alpha_min
        beta_new = num10 / denom1 if denom1 > 0 else beta_min

        alpha_new = float(np.clip(alpha_new, alpha_min, alpha_max))
        beta_new = float(np.clip(beta_new, beta_min, 1 - eps))

        delta = max(
            np.max(np.abs(theta_new - theta)),
            abs(alpha_new - alpha),
            abs(beta_new - beta),
        )

        if it == 0 or (it + 1) % 10 == 0:
            checkpoint(
                f"[EM] iter={it+1:03d}  loglik={loglik:.6f}  "
                f"delta={delta:.3e}  alpha={alpha_new:.4f}  beta={beta_new:.4f}",
                verbose,
            )

        theta = theta_new
        alpha = alpha_new
        beta = beta_new

        if it > 0 and abs(loglik_trace[-1] - loglik_trace[-2]) < tol and delta < tol:
            checkpoint(f"[EM] stop at iter={it+1}, loglik={loglik:.6f}", verbose)
            break

    T = emission_matrix_from_alpha_beta(
        latent_states=latent_states,
        observed_states=observed_states,
        alpha=alpha,
        beta=beta,
    )
    obs_probs = np.clip(pi_new @ T, 1e-300, None)

    M_common = np.array([
        [1 - alpha, alpha],
        [beta, 1 - beta],
    ])

    checkpoint("=== END EM ===", verbose)

    return {
        "theta_hat": theta,
        "y_hat": np.exp(theta),
        "latent_probs_hat": pi_new,
        "alpha_hat": alpha,
        "beta_hat": beta,
        "M_hat": M_common,
        "transition_hat_statewise": T,
        "obs_probs_hat": obs_probs,
        "expected_obs_counts": observed_counts.sum() * obs_probs,
        "loglik": float(np.sum(observed_counts * np.log(obs_probs))),
        "n_iter": len(loglik_trace),
        "loglik_trace": loglik_trace,
    }


# ============================================================
# POSTERIOR I KOREKTA PO EM
# ============================================================

def posterior_latent_given_observed(fit_em):
    pi_hat = np.asarray(fit_em["latent_probs_hat"], dtype=float)
    T_hat = np.asarray(fit_em["transition_hat_statewise"], dtype=float)
    obs_probs_hat = np.asarray(fit_em["obs_probs_hat"], dtype=float)

    post = (pi_hat[:, None] * T_hat) / np.clip(obs_probs_hat[None, :], 1e-300, None)
    post = post / post.sum(axis=0, keepdims=True)
    return post


def latent_bit_priors_from_fit(fit_em, latent_states):
    """
    prior_true_j = P(Z_j = 1) pod wyestymowanym latentnym multG
    """
    return fit_em["latent_probs_hat"] @ latent_states


def correct_bits_independently_from_em_params(X_obs, alpha, beta, prior_true):
    """
    Czysta korekta bit-po-bicie:
    dla każdego bitu osobno używamy tylko:
      - X_j
      - alpha
      - beta
      - prior_true_j = P(Z_j=1)

    Nie używamy posterioru po całych stanach do ustawiania bitów.
    """
    X_obs = np.asarray(X_obs, dtype=int)
    n, p = X_obs.shape
    X_corr = np.zeros_like(X_obs)

    for j in range(p):
        pi = prior_true[j]

        # P(Z_j=1 | X_j=1)
        post1 = ((1 - beta) * pi) / (((1 - beta) * pi) + alpha * (1 - pi))

        # P(Z_j=1 | X_j=0)
        post0 = (beta * pi) / ((beta * pi) + (1 - alpha) * (1 - pi))

        X_corr[X_obs[:, j] == 1, j] = int(post1 > 0.5)
        X_corr[X_obs[:, j] == 0, j] = int(post0 > 0.5)

    return X_corr


# ============================================================
# STATYSTYKA TESTOWA
# ============================================================

def deviance_statistic(counts, probs):
    counts = np.asarray(counts, dtype=float)
    probs = np.asarray(probs, dtype=float)

    n = counts.sum()
    expected = np.clip(n * probs, 1e-300, None)

    mask = counts > 0
    return float(2.0 * np.sum(counts[mask] * np.log(counts[mask] / expected[mask])))


# ============================================================
# SYMULACJA
# ============================================================

def simulate_from_em_model(n, latent_states, latent_probs, alpha, beta, rng):
    K, p = latent_states.shape

    idx = rng.choice(K, size=n, p=latent_probs)
    Z = latent_states[idx]
    X = np.zeros_like(Z)

    for j in range(p):
        u = rng.random(n)
        z = Z[:, j]

        mask0 = (z == 0)
        X[mask0, j] = (u[mask0] < alpha).astype(int)

        mask1 = (z == 1)
        X[mask1, j] = (u[mask1] >= beta).astype(int)

    return Z, X


# ============================================================
# TEST 1: PO KOREKCIE BIT-PO-BICIE I OBCIĘCIU DO NOŚNIKA
# ============================================================

def test_after_bitwise_correction_and_cut(
    X_obs,
    fit_em,
    latent_masks,
    latent_states,
    bootstrap_B=300,
    refit_multg_in_bootstrap=True,
    seed=123,
    verbose=VERBOSE,
):
    checkpoint("=== TEST CUT: czysta korekta bit-po-bicie i obcięcie do nośnika ===", verbose)

    alpha = fit_em["alpha_hat"]
    beta = fit_em["beta_hat"]
    prior_true = latent_bit_priors_from_fit(fit_em, latent_states)

    X_corr = correct_bits_independently_from_em_params(
        X_obs=X_obs,
        alpha=alpha,
        beta=beta,
        prior_true=prior_true,
    )

    is_valid = valid_rows_mask(X_corr, EDGES)
    X_valid = X_corr[is_valid]

    checkpoint(
        f"[CUT] n_total={len(X_obs)}, n_valid={len(X_valid)}, n_invalid={len(X_obs)-len(X_valid)}",
        verbose,
    )

    if len(X_valid) == 0:
        raise ValueError("Po korekcie i obcięciu do nośnika nie zostały żadne dane.")

    latent_mask_to_index = {int(m): i for i, m in enumerate(latent_masks.tolist())}
    valid_masks = matrix_to_masks(X_valid)

    counts_cut = np.zeros(len(latent_masks), dtype=int)
    for m in valid_masks:
        counts_cut[latent_mask_to_index[int(m)]] += 1

    fit_mult = fit_multG1_from_state_counts(
        state_counts=counts_cut,
        support_states=latent_states,
    )

    D_obs = deviance_statistic(counts_cut, fit_mult["probs_hat"])
    checkpoint(f"[CUT] D_obs={D_obs:.6f}", verbose)

    rng = np.random.default_rng(seed)
    D_boot = np.empty(bootstrap_B, dtype=float)

    for b in range(bootstrap_B):
        boot_counts = rng.multinomial(len(X_valid), fit_mult["probs_hat"])

        if refit_multg_in_bootstrap:
            fit_b = fit_multG1_from_state_counts(
                state_counts=boot_counts,
                support_states=latent_states,
                theta_init=fit_mult["theta_hat"],
            )
            D_boot[b] = deviance_statistic(boot_counts, fit_b["probs_hat"])
        else:
            D_boot[b] = deviance_statistic(boot_counts, fit_mult["probs_hat"])

        if (b + 1) % 50 == 0 or b == 0 or b + 1 == bootstrap_B:
            checkpoint(f"[CUT bootstrap] {b+1}/{bootstrap_B}", verbose)

    p_value = (1.0 + np.sum(D_boot >= D_obs)) / (bootstrap_B + 1.0)
    checkpoint(f"[CUT] p_value={p_value:.6f}", verbose)

    state_table = pd.DataFrame({
        "latent_mask": latent_masks,
        "latent_state": [mask_to_bitstring(int(m), P) for m in latent_masks],
        "obs_after_correction_cut": counts_cut,
        "exp_under_multG": fit_mult["expected_state_counts"],
        "prob_hat_multG": fit_mult["probs_hat"],
    }).sort_values(["obs_after_correction_cut", "exp_under_multG"], ascending=[False, False]).reset_index(drop=True)

    return {
        "summary": {
            "n_total": int(len(X_obs)),
            "n_valid_after_cut": int(len(X_valid)),
            "n_invalid_after_cut": int(len(X_obs) - len(X_valid)),
            "frac_valid_after_cut": float(len(X_valid) / len(X_obs)),
            "deviance_obs_cut": float(D_obs),
            "bootstrap_p_value_cut": float(p_value),
            "reject_multG_at_05_cut": bool(p_value < 0.05),
            "optimizer_success_cut": bool(fit_mult["optimizer_success"]),
            "optimizer_message_cut": fit_mult["optimizer_message"],
        },
        "X_corr": X_corr,
        "X_valid": X_valid,
        "valid_rows_mask_after_cut": is_valid,
        "fit_mult_cut": fit_mult,
        "state_table_cut": state_table,
        "D_boot_cut": D_boot,
    }


# ============================================================
# TEST 2: BEZ OBCINANIA NOŚNIKA
# ============================================================

def test_without_cutting_support(
    X_obs,
    observed_counts,
    observed_states,
    latent_states,
    fit_em,
    bootstrap_B=300,
    refit_em_in_bootstrap=True,
    seed=123,
    verbose=VERBOSE,
):
    checkpoint("=== TEST NO-CUT: pełny model noisy multG ===", verbose)

    D_obs = deviance_statistic(observed_counts, fit_em["obs_probs_hat"])
    checkpoint(f"[NO-CUT] D_obs={D_obs:.6f}", verbose)

    rng = np.random.default_rng(seed)
    D_boot = np.empty(bootstrap_B, dtype=float)

    for b in range(bootstrap_B):
        _, Xb = simulate_from_em_model(
            n=len(X_obs),
            latent_states=latent_states,
            latent_probs=fit_em["latent_probs_hat"],
            alpha=fit_em["alpha_hat"],
            beta=fit_em["beta_hat"],
            rng=rng,
        )

        _, obs_states_b, obs_counts_b = counts_over_all_patterns(Xb)

        if refit_em_in_bootstrap:
            fit_b = em_fit_multG_bitflip(
                observed_counts=obs_counts_b,
                observed_states=obs_states_b,
                latent_states=latent_states,
                alpha_init=fit_em["alpha_hat"],
                beta_init=fit_em["beta_hat"],
                max_iter=EM_MAX_ITER,
                tol=EM_TOL,
                eps=EM_EPS,
                alpha_min=ALPHA_MIN,
                alpha_max=ALPHA_MAX,
                beta_min=BETA_MIN,
                verbose=False,
            )
            D_boot[b] = deviance_statistic(obs_counts_b, fit_b["obs_probs_hat"])
        else:
            D_boot[b] = deviance_statistic(obs_counts_b, fit_em["obs_probs_hat"])

        if (b + 1) % 25 == 0 or b == 0 or b + 1 == bootstrap_B:
            checkpoint(f"[NO-CUT bootstrap] {b+1}/{bootstrap_B}", verbose)

    p_value = (1.0 + np.sum(D_boot >= D_obs)) / (bootstrap_B + 1.0)
    checkpoint(f"[NO-CUT] p_value={p_value:.6f}", verbose)

    obs_table = pd.DataFrame({
        "obs_state": [mask_to_bitstring(i, P) for i in range(2 ** P)],
        "obs_count": observed_counts,
        "exp_under_noisy_model": fit_em["expected_obs_counts"],
        "obs_prob_hat": fit_em["obs_probs_hat"],
    }).sort_values(["obs_count", "exp_under_noisy_model"], ascending=[False, False]).reset_index(drop=True)

    return {
        "summary": {
            "deviance_obs_no_cut": float(D_obs),
            "bootstrap_p_value_no_cut": float(p_value),
            "reject_model_at_05_no_cut": bool(p_value < 0.05),
        },
        "obs_table_no_cut": obs_table,
        "D_boot_no_cut": D_boot,
    }


# ============================================================
# GŁÓWNA FUNKCJA
# ============================================================

def run_full_pipeline_em_bitflip_and_two_tests(
    csv_path,
    alpha_init,
    beta_init,
    bootstrap_B=300,
    refit_em_in_bootstrap=True,
    refit_multg_in_bootstrap=True,
    seed=123,
    verbose=VERBOSE,
):
    checkpoint("=== START PIPELINE ===", verbose)

    X_obs = read_experiment_data(csv_path)
    checkpoint(f"[PIPELINE] loaded X_obs with n={len(X_obs)}", verbose)

    obs_masks, observed_states, observed_counts = counts_over_all_patterns(X_obs)

    latent_masks = enumerate_independent_set_masks(P, EDGES)
    latent_states = masks_to_matrix(latent_masks, P)
    checkpoint(f"[PIPELINE] latent support size={len(latent_masks)}", verbose)

    fit_em = em_fit_multG_bitflip(
        observed_counts=observed_counts,
        observed_states=observed_states,
        latent_states=latent_states,
        alpha_init=alpha_init,
        beta_init=beta_init,
        max_iter=EM_MAX_ITER,
        tol=EM_TOL,
        eps=EM_EPS,
        alpha_min=ALPHA_MIN,
        alpha_max=ALPHA_MAX,
        beta_min=BETA_MIN,
        verbose=verbose,
    )

    test_cut = test_after_bitwise_correction_and_cut(
        X_obs=X_obs,
        fit_em=fit_em,
        latent_masks=latent_masks,
        latent_states=latent_states,
        bootstrap_B=bootstrap_B,
        refit_multg_in_bootstrap=refit_multg_in_bootstrap,
        seed=seed,
        verbose=verbose,
    )

    test_no_cut = test_without_cutting_support(
        X_obs=X_obs,
        observed_counts=observed_counts,
        observed_states=observed_states,
        latent_states=latent_states,
        fit_em=fit_em,
        bootstrap_B=bootstrap_B,
        refit_em_in_bootstrap=refit_em_in_bootstrap,
        seed=seed,
        verbose=verbose,
    )

    checkpoint("=== END PIPELINE ===", verbose)

    latent_table = pd.DataFrame({
        "latent_mask": latent_masks,
        "latent_state": [mask_to_bitstring(int(m), P) for m in latent_masks],
        "latent_prob_hat": fit_em["latent_probs_hat"],
    }).sort_values("latent_prob_hat", ascending=False).reset_index(drop=True)

    return {
        "summary": {
            "csv_path": csv_path,
            "graph": GRAPH_NAME,
            "edges_python": EDGES,
            "n_total": int(len(X_obs)),
            "support_size": int(len(latent_masks)),
            "em_loglik": float(fit_em["loglik"]),
            "em_iterations": int(fit_em["n_iter"]),
            "beta_min_used": float(BETA_MIN),
            "alpha_max_used": float(ALPHA_MAX),
        },
        "fit_em": fit_em,
        "latent_table": latent_table,
        "test_cut": test_cut,
        "test_no_cut": test_no_cut,
    }


def print_full_pipeline_report(result, top_k=20, digits=4):
    s = result["summary"]
    sc = result["test_cut"]["summary"]
    sn = result["test_no_cut"]["summary"]

    print("=== EM + DWA TESTY ===")
    print(f"csv_path                = {s['csv_path']}")
    print(f"graph                   = {s['graph']}")
    print(f"edges_python            = {s['edges_python']}")
    print(f"n_total                 = {s['n_total']}")
    print(f"support_size            = {s['support_size']}")
    print(f"em_loglik               = {s['em_loglik']:.4f}")
    print(f"em_iterations           = {s['em_iterations']}")
    print(f"beta_min_used           = {s['beta_min_used']:.4f}")
    print(f"alpha_max_used          = {s['alpha_max_used']:.4f}")
    print()

    print("=== WSPÓLNA BITOWA MACIERZ M ===")
    print(pd.DataFrame(
        result["fit_em"]["M_hat"],
        index=["true0", "true1"],
        columns=["obs0", "obs1"]
    ).round(digits).to_string())
    print()
    print(f"alpha_hat = {result['fit_em']['alpha_hat']:.4f}")
    print(f"beta_hat  = {result['fit_em']['beta_hat']:.4f}")
    print()

    print("=== TEST 1: PO KOREKCIE I OBCIĘCIU DO NOŚNIKA ===")
    print(f"n_valid_after_cut       = {sc['n_valid_after_cut']}")
    print(f"n_invalid_after_cut     = {sc['n_invalid_after_cut']}")
    print(f"frac_valid_after_cut    = {sc['frac_valid_after_cut']:.4f}")
    print(f"deviance_obs_cut        = {sc['deviance_obs_cut']:.4f}")
    print(f"bootstrap_p_value_cut   = {sc['bootstrap_p_value_cut']:.4f}")
    print(f"reject_multG_at_05_cut  = {sc['reject_multG_at_05_cut']}")
    print()

    print("=== TEST 2: BEZ OBCINANIA NOŚNIKA ===")
    print(f"deviance_obs_no_cut      = {sn['deviance_obs_no_cut']:.4f}")
    print(f"bootstrap_p_value_no_cut = {sn['bootstrap_p_value_no_cut']:.4f}")
    print(f"reject_model_at_05_no_cut= {sn['reject_model_at_05_no_cut']}")
    print()

    print(f"=== TOP {top_k} LATENTNYCH STANÓW ===")
    print(result["latent_table"].head(top_k).round(digits).to_string(index=False))
    print()

    print(f"=== TOP {top_k} STANÓW PO KOREKCIE I OBCIĘCIU ===")
    print(result["test_cut"]["state_table_cut"].head(top_k).round(digits).to_string(index=False))
    print()

    print(f"=== TOP {top_k} STANÓW OBSERWOWANYCH (NO-CUT) ===")
    print(result["test_no_cut"]["obs_table_no_cut"].head(top_k).round(digits).to_string(index=False))


# ============================================================
# URUCHOMIENIE
# ============================================================

result = run_full_pipeline_em_bitflip_and_two_tests(
    csv_path=CSV_PATH,
    alpha_init=ALPHA_INIT,
    beta_init=BETA_INIT,
    bootstrap_B=BOOTSTRAP_B,
    refit_em_in_bootstrap=REFIT_EM_IN_BOOTSTRAP,
    refit_multg_in_bootstrap=REFIT_MULTG_IN_BOOTSTRAP,
    seed=SEED,
    verbose=True,
)

print_full_pipeline_report(result, top_k=25, digits=4)

=== START PIPELINE ===
[PIPELINE] loaded X_obs with n=678
[PIPELINE] latent support size=9
=== START EM ===
K=9, M=16, p=4
alpha_init=0.1000
beta_init =0.0500
alpha_max =0.5
beta_min  =0.03
[EM] iter=001  loglik=-2458.045032  delta=1.449e+00  alpha=0.3826  beta=0.0300
[EM] iter=010  loglik=-1832.728970  delta=3.041e-01  alpha=0.5000  beta=0.1443
[EM] iter=020  loglik=-1824.394454  delta=2.420e-01  alpha=0.5000  beta=0.2250
[EM] iter=030  loglik=-1824.362111  delta=2.373e-01  alpha=0.5000  beta=0.2296
[EM] iter=040  loglik=-1824.362066  delta=2.372e-01  alpha=0.5000  beta=0.2298
[EM] iter=050  loglik=-1824.362066  delta=2.372e-01  alpha=0.5000  beta=0.2298
[EM] iter=060  loglik=-1824.362066  delta=2.372e-01  alpha=0.5000  beta=0.2298
[EM] iter=070  loglik=-1824.362066  delta=2.372e-01  alpha=0.5000  beta=0.2298
[EM] iter=080  loglik=-1824.362066  delta=2.372e-01  alpha=0.5000  beta=0.2298
[EM] iter=090  loglik=-1824.362066  delta=2.372e-01  alpha=0.5000  beta=0.2298
[EM] iter=100  logli